In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/linzkibet/frauddetectiodata/Synthetic Mobile Money Transaction Dataset/MoMTSim_20240722202413_1000_dataset.csv


In [1]:
# Import pandas for loading and analysing the data
import pandas as pd

# Full path to the MoMTSim file in the Kaggle input directory
path = "/kaggle/input/datasets/linzkibet/frauddetectiodata/Synthetic Mobile Money Transaction Dataset/MoMTSim_20240722202413_1000_dataset.csv"

# Load the CSV into a DataFrame (this may take a minute for ~1.7M rows)
df = pd.read_csv(path)

# Total number of transactions; ~1.72M suggests version 1, ~4.2M suggests version 2
print("Rows:", len(df))

# Column names, needed to set the variables in Cell 2
print("Columns:", list(df.columns))

# Data type of each column (checks numbers were read as numbers)
print(df.dtypes)

# First five rows, to see what a transaction looks like
print(df.head())

# Count of missing values per column
print("Nulls:\n", df.isna().sum())

Rows: 4225958
Columns: ['step', 'transactionType', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'recipient', 'oldBalRecipient', 'newBalRecipient', 'isFraud']
step                 int64
transactionType     object
amount             float64
initiator            int64
oldBalInitiator    float64
newBalInitiator    float64
recipient           object
oldBalRecipient    float64
newBalRecipient    float64
isFraud              int64
dtype: object
   step transactionType  amount         initiator  oldBalInitiator  \
0     0         PAYMENT  579.16  4018807983755588            29.32   
1     0        TRANSFER   39.03  4610593854752035            39.03   
2     0        TRANSFER   21.78  4840578447365454            21.78   
3     0         PAYMENT  570.22  4958066799829630             5.29   
4     0         PAYMENT  574.80  4149196095918843            82.63   

   newBalInitiator         recipient  oldBalRecipient  newBalRecipient  \
0            29.32        30-0000345           

In [3]:
# Column names taken from the actual header printed in Cell 1
fraud_col = "isFraud"           # fraud label (1 = fraud, 0 = legitimate)
type_col  = "transactionType"   # transaction type (PAYMENT, TRANSFER, etc.)
orig_col  = "initiator"         # account that initiates the transaction
dest_col  = "recipient"         # recipient account or merchant/institution code
step_col  = "step"              # time index of the transaction
old_bal   = "oldBalInitiator"   # initiator's balance before the transaction
new_bal   = "newBalInitiator"   # initiator's balance after the transaction

# 1. Overall fraud rate (mean of a 0/1 column is the proportion of fraud)
print("Fraud rate:", df[fraud_col].mean())

# 2. Fraud by transaction type: count, number of fraud cases, and fraud rate
#    Shows which transaction types carry the fraud
print(df.groupby(type_col)[fraud_col].agg(["count", "sum", "mean"]))

# 3. Number of transactions per initiating account
#    This decides whether per-customer behavioural baselines are viable
per_acct = df.groupby(orig_col).size()
print("Distinct initiators:", len(per_acct))
print(per_acct.describe())

# Share of accounts with at least k transactions, for several values of k
for k in (3, 5, 10, 20):
    print(f"Accounts with >= {k} txns: {(per_acct >= k).mean():.1%}")

# 4. Check whether the same accounts and recipients repeat in fraud rows
fraud = df[df[fraud_col] == 1]   # subset containing only fraudulent transactions
print("Fraud rows:", len(fraud))
print("Distinct initiators in fraud rows:", fraud[orig_col].nunique())
print("Distinct recipients in fraud rows:", fraud[dest_col].nunique())

# 5. Leakage check: how often the initiator's balance ends at zero,
#    compared between fraud rows and legitimate rows
legit = df[df[fraud_col] == 0]   # subset containing only legitimate transactions
print("Fraud rows with new initiator balance = 0:", (fraud[new_bal] == 0).mean())
print("Legit rows with new initiator balance = 0:", (legit[new_bal] == 0).mean())

# Stronger leakage signal: the transaction amount equals the full balance
# (the account is completely drained), as seen in the first rows of the data
print("Fraud rows where amount == old balance:", (fraud["amount"] == fraud[old_bal]).mean())
print("Legit rows where amount == old balance:", (legit["amount"] == legit[old_bal]).mean())

# 6. Time range of the data (needed later for the temporal train/test split)
print(df[step_col].describe())

# 7. Do fraudulent and legitimate transactions differ in amount?
#    Median and mean amount by label
print(df.groupby(fraud_col)["amount"].agg(["median", "mean", "max"]))

Fraud rate: 0.5284288201633807
                   count      sum      mean
transactionType                            
DEBIT               1325        0  0.000000
DEPOSIT             8824        0  0.000000
PAYMENT          1677798        0  0.000000
TRANSFER         2510947  2233118  0.889353
WITHDRAWAL         27064        0  0.000000
Distinct initiators: 1653981
count    1.653981e+06
mean     2.555022e+00
std      6.004262e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      9.085000e+03
dtype: float64
Accounts with >= 3 txns: 0.2%
Accounts with >= 5 txns: 0.2%
Accounts with >= 10 txns: 0.2%
Accounts with >= 20 txns: 0.2%
Fraud rows: 2233118
Distinct initiators in fraud rows: 1653020
Distinct recipients in fraud rows: 19000
Fraud rows with new initiator balance = 0: 0.738860194579955
Legit rows with new initiator balance = 0: 0.22914584211477088
Fraud rows where amount == old balance: 0.7388597467755846
Legit rows where amount == old b

This output settles several design questions, and a few of them change the plan.

What the audit shows

Finding	What it means
4,225,958 rows, 52.84% fraud	This is version 2. It is the only file you have, so it becomes the working dataset. Down-sample fraud for the realistic-prevalence test.
All fraud is in TRANSFER (2,233,118 of 2,510,947 transfers, 88.9%)	PAYMENT, DEBIT, DEPOSIT and WITHDRAWAL have zero fraud. The real modelling problem is separating fraudulent from legitimate transfers.
Median 1 transaction per initiator; only 0.2% of accounts have 3 or more	Per-customer baselines are not viable for about 99.8% of accounts. This was the main risk flagged earlier, and it has materialised.
2.23M fraud rows come from 1.65M distinct initiators	Fraud victims appear almost once each, so there is no history to compare against.
Fraud goes to only 19,000 distinct recipients	Roughly 117 fraud transfers per recipient. Recipient-side behaviour (many senders, high inflow) is a strong and usable signal.
73.9% of fraud rows have amount equal to the old balance; legitimate rows: 0.0002%	Severe leakage. The rule “amount equals the full balance” alone catches about 74% of fraud at near-perfect precision.
Median amount: fraud 191.75, legitimate 597.27; legitimate max about 765 million	Fraud is not defined by large amounts, so a fixed amount threshold will perform poorly. The legitimate side has extreme outliers, so use log-scaled amounts.
step runs 0 to 192	About 8 days if each step is an hour (confirm in the MoMTSim documentation). This allows an hour-of-day feature and a temporal split.

Consequences for the proposal

Reframe the behavioural claim. This dataset cannot demonstrate “unusual for this customer.” It can demonstrate peer-group deviation (amount against the typical transfer) and recipient-level behaviour (a recipient suddenly receiving from many distinct senders). Per-customer baselines need real SACCO exports, which strengthens the case for the pilot stage. State this in the evaluation report.
Alert text changes. The “typical transfer amount is KSh 2,000” example cannot be generated for most accounts here. Explanations would cite peer-group comparisons and recipient activity instead.
The drain pattern is an artefact. An alert reading “transaction emptied the account” would be technically correct but reflects how the simulator injects fraud. Run every model with and without the balance features and report both.
Recipient identity must not leak. If the model memorises the 19,000 fraud recipient IDs, it will score well on a random split and fail on new recipients. Recipient features must be computed only from earlier steps, and the split must be temporal.

In [4]:
# Restrict to TRANSFER rows, since all fraud occurs in this transaction type
tr = df[df[type_col] == "TRANSFER"]

# 1. Profile the small group of initiators with 3 or more transactions
heavy_ids = per_acct[per_acct >= 3].index          # ids of the repeat initiators
heavy = df[df[orig_col].isin(heavy_ids)]           # all rows belonging to them
print("Heavy initiators:", len(heavy_ids))
print("Share of all rows they account for:", len(heavy) / len(df))
print("Transaction types used by heavy initiators:")
print(heavy[type_col].value_counts())              # which types they perform
print("Fraud rate among heavy initiators:", heavy[fraud_col].mean())

# 2. Do fraud recipients also receive legitimate transfers?
fraud_recips = set(tr[tr[fraud_col] == 1][dest_col].unique())   # recipients seen in fraud
legit_tr = tr[tr[fraud_col] == 0]                                # legitimate transfers
overlap = legit_tr[legit_tr[dest_col].isin(fraud_recips)]        # legit transfers to those recipients
print("Legit transfers going to fraud recipients:", len(overlap), "of", len(legit_tr))
print("Distinct recipients in legit transfers:", legit_tr[dest_col].nunique())

# 3. Recipient identifier format: customer-style numeric ids vs coded recipients
#    (for example "30-0000345"). Shows whether the format separates fraud from legitimate.
tr_fmt = tr.assign(recip_is_coded=tr[dest_col].astype(str).str.contains("-"))   # True if the id has a hyphen
print(tr_fmt.groupby("recip_is_coded")[fraud_col].agg(["count", "mean"]))

# 4. Strength of the drain rule as a baseline on TRANSFER rows
#    Rule: flag a transfer if the amount equals the initiator's full old balance
rule = (tr["amount"] == tr[old_bal])                  # boolean flag per transfer
tp = (rule & (tr[fraud_col] == 1)).sum()              # frauds caught by the rule
fp = (rule & (tr[fraud_col] == 0)).sum()              # legitimate transfers wrongly flagged
fn = ((~rule) & (tr[fraud_col] == 1)).sum()           # frauds missed by the rule
print("Drain rule precision:", tp / (tp + fp))
print("Drain rule recall:", tp / (tp + fn))

# 5. Fraud rate by hour of day, assuming each step is one hour
tr_hour = tr.assign(hour=tr[step_col] % 24)           # hour-of-day feature
print(tr_hour.groupby("hour")[fraud_col].mean().round(3))

Heavy initiators: 3048
Share of all rows they account for: 0.6091776113250534
Transaction types used by heavy initiators:
transactionType
PAYMENT       1676292
TRANSFER       860862
WITHDRAWAL      27063
DEPOSIT          8817
DEBIT            1325
Name: count, dtype: int64
Fraud rate among heavy initiators: 0.2264773483418591
Legit transfers going to fraud recipients: 270415 of 277829
Distinct recipients in legit transfers: 11043
                  count      mean
recip_is_coded                   
False           2510947  0.889353
Drain rule precision: 1.0
Drain rule recall: 0.7388597467755846
hour
0     0.886
1     0.899
2     0.899
3     0.898
4     0.897
5     0.898
6     0.899
7     0.897
8     0.895
9     0.897
10    0.897
11    0.896
12    0.894
13    0.891
14    0.888
15    0.888
16    0.886
17    0.883
18    0.880
19    0.875
20    0.877
21    0.874
22    0.877
23    0.875
Name: isFraud, dtype: float64


In [5]:
# TRANSFER rows only, since all fraud occurs in this transaction type
tr = df[df[type_col] == "TRANSFER"].copy()

# Flag initiators with 3 or more transactions (heavy_ids comes from the previous cell)
tr["heavy"] = tr[orig_col].isin(heavy_ids)

# Flag the drain pattern: amount equals the initiator's full old balance
tr["drain"] = (tr["amount"] == tr[old_bal])

# 1. Confirm the two fraud populations by cross-tabulating fraud
#    against history (heavy) and the drain pattern
print(pd.crosstab([tr["heavy"], tr["drain"]], tr[fraud_col]))

# 2. Are the heavy accounts a mix, or split into fraud-only and legitimate-only accounts?
heavy_tr = tr[tr["heavy"]]                                   # transfers from history-rich accounts
acct_fraud = heavy_tr.groupby(orig_col)[fraud_col].agg(
    n_transfers="count",                                     # transfers per account
    fraud_rate="mean"                                        # share of that account's transfers that are fraud
)
print(acct_fraud.describe())

# Number of accounts in each fraud-rate band; a split between the first
# and last bands suggests separate fraudster and customer accounts
bands = pd.cut(acct_fraud["fraud_rate"], bins=[-0.01, 0, 0.1, 0.5, 0.9, 1.0])
print(bands.value_counts().sort_index())

# 3. How many heavy accounts have any legitimate transfers (needed for a baseline)?
print("Heavy accounts with legit transfers:",
      heavy_tr[heavy_tr[fraud_col] == 0][orig_col].nunique(), "of", len(heavy_ids))

# 4. Prototype of a per-account deviation feature (exploratory only: the median
#    uses the whole time range, so the final version must use past data only)
legit_med = (heavy_tr[heavy_tr[fraud_col] == 0]
             .groupby(orig_col)["amount"].median()
             .rename("acct_median_amt"))                     # each account's typical legitimate transfer
h = heavy_tr.join(legit_med, on=orig_col)                    # attach the median to every transfer
h = h[h["acct_median_amt"] > 0].copy()                       # drop accounts with no legit baseline or a zero median
h["amt_ratio"] = h["amount"] / h["acct_median_amt"]          # amount relative to the account's own norm

# Compare the deviation for legitimate (0) and fraudulent (1) transfers
print(h.groupby(fraud_col)["amt_ratio"].describe())

# Same comparison restricted to non-drain fraud (population B) against legitimate transfers
print(h[~h["drain"]].groupby(fraud_col)["amt_ratio"].describe())

isFraud           0        1
heavy drain                 
False False       1      390
      True        0  1649694
True  False  277828   582767
      True        0      267
       n_transfers   fraud_rate
count  2975.000000  2975.000000
mean    289.365378     0.790785
std     393.310426     0.330499
min       1.000000     0.000000
25%      18.000000     0.327077
50%     141.000000     1.000000
75%     424.000000     1.000000
max    3775.000000     1.000000
fraud_rate
(-0.01, 0.0]      39
(0.0, 0.1]         5
(0.1, 0.5]       813
(0.5, 0.9]         5
(0.9, 1.0]      2113
Name: count, dtype: int64
Heavy accounts with legit transfers: 862 of 3048
            count      mean       std       min       25%       50%       75%  \
isFraud                                                                         
0        277828.0  1.725386  5.213529  0.009345  0.986923  1.000000  1.041667   
1        115243.0  1.046808  0.149688  0.039962  0.986520  0.998628  1.035503   

                max  


The results change the scope of the modelling data, so I’ll cover those first and then the preparation steps.

What the output shows

Finding	What it means
Accounts with fewer than 3 transactions generate 1,650,085 transfers, and only 1 is legitimate	There are no negatives to learn from or evaluate against for new accounts. They cannot go into a trained model. They can only be described, for example “new account, drained balance.”
All 277,828 other legitimate transfers come from history-rich accounts, which also carry 583,034 fraud rows	The usable modelling problem is fraud against legitimate transfers within this group.
2,113 accounts have more than 90% fraud, and 862 accounts contain all the legitimate transfers	Fraud here is largely driven by fraud-dedicated accounts. Roughly 468,000 of the 583,000 fraud rows come from them. Only about 115,000 come from accounts that also make legitimate transfers.
Amount against the account’s own norm: fraud median 0.9986, maximum 1.95. Legitimate: median 1.0, maximum 194	Fraud amounts look like the account’s usual amounts. Unusually large amounts occur only in legitimate transfers.

The last finding contradicts the proposal’s example, where an unusually large amount signals fraud. In this dataset the opposite holds, which also gives you a “legitimate but unusual” test subset: legitimate transfers with a high deviation from the account’s norm. A fixed or deviation-based rule would flag them, so the test is whether your model avoids doing so.

My earlier prototype computed each account’s median from legitimate transfers only. That used the label, so it was exploratory. The features below use only earlier transactions and never the label.

Your inputs

The target is isFraud. The model scope is TRANSFER rows from initiators with at least 5 earlier transfers. Within that scope every history-based feature is defined for every row, so a missing value cannot reveal that an account is new.

Input set	Features	Role
Behavioural (headline)	amount_log, hour, amt_logratio_own, amt_z_own, prior_mean_log, prior_std_log, steps_since_last, is_new_recipient, recip_prior_count	The model you report as the behavioural system
Balance fields	oldBalInitiator_log, newBalInitiator_log, amount_to_balance, drains_account, init_bal_diff, oldBalRecipient_log, newBalRecipient_log, recip_bal_diff	Leakage ablation: report results with and without
Tenure	prior_count	Ablation only

The columns initiator, recipient, row_id, step and transactionType are not inputs. Identifiers are used only to compute features and are then dropped, which also supports data minimisation.

**Cell 1: Scope and cleaning (steps 3 and 4)**

In [6]:
# Import the libraries used throughout the preparation steps
import numpy as np
import pandas as pd

# Keep TRANSFER rows only, because every fraud case in this dataset is a TRANSFER
tr = df[df["transactionType"] == "TRANSFER"].copy()

# Keep the original row position so each row can be traced back to the raw file
tr = tr.rename_axis("row_id").reset_index()

# Identifiers are a mix of numbers and codes, so store them as strings
tr["initiator"] = tr["initiator"].astype(str)
tr["recipient"] = tr["recipient"].astype(str)

# Step 3: cleaning checks
print("Rows before cleaning:", len(tr))                                          # starting size
dup_mask = tr.drop(columns="row_id").duplicated()                                # duplicates, ignoring row_id
print("Exact duplicate rows:", dup_mask.sum())
print("Amount <= 0:", (tr["amount"] <= 0).sum())                                 # empty or impossible transfers

# Negative balances would indicate corrupted records
bal_cols = ["oldBalInitiator", "newBalInitiator", "oldBalRecipient", "newBalRecipient"]
print("Negative balances per column:\n", (tr[bal_cols] < 0).sum())

# Drop exact duplicates and non-positive amounts; outliers are deliberately kept
tr = tr[~dup_mask & (tr["amount"] > 0)].reset_index(drop=True)
print("Rows after cleaning:", len(tr))

Rows before cleaning: 2510947
Exact duplicate rows: 0
Amount <= 0: 58
Negative balances per column:
 oldBalInitiator    8398
newBalInitiator    8915
oldBalRecipient       1
newBalRecipient       0
dtype: int64
Rows after cleaning: 2510889


**Cell 2: Transformation and feature engineering (steps 4 and 5)**

In [8]:
# Log-scale the amount, because amounts range from cents to hundreds of millions
tr["amount_log"] = np.log1p(tr["amount"])
tr["amount_log_sq"] = tr["amount_log"] ** 2        # helper column for the running variance

# Hour of day, assuming each step is one hour (confirm in the MoMTSim documentation)
tr["hour"] = tr["step"] % 24

# Sort by initiator, then time, so every running statistic sees only earlier transfers
tr = tr.sort_values(["initiator", "step", "row_id"], kind="stable").reset_index(drop=True)

g = tr.groupby("initiator")                         # one group per initiating account

# Number of earlier transfers by the same initiator (0 for the first transfer)
tr["prior_count"] = g.cumcount()
n = tr["prior_count"].where(tr["prior_count"] > 0)  # same count, NaN when there is no history

# Sums over earlier rows only: running total minus the current row
s1 = g["amount_log"].cumsum() - tr["amount_log"]        # sum of earlier log amounts
s2 = g["amount_log_sq"].cumsum() - tr["amount_log_sq"]  # sum of earlier squared log amounts

# Account's typical amount and its variability, from earlier transfers only
tr["prior_mean_log"] = s1 / n
var = (s2 / n) - tr["prior_mean_log"] ** 2               # running variance
tr["prior_std_log"] = np.sqrt(var.clip(lower=0))         # clip guards against tiny negative rounding errors

# Deviation of this transfer from the account's own norm
Z_FLOOR = 0.1                                            # floor so near-constant accounts do not give huge z-scores
tr["amt_logratio_own"] = tr["amount_log"] - tr["prior_mean_log"]
tr["amt_z_own"] = tr["amt_logratio_own"] / (tr["prior_std_log"] + Z_FLOOR)

# Time since the account's previous transfer, in steps
tr["steps_since_last"] = g["step"].diff()

# 1 if this is the first time this initiator sends to this recipient (rows are in time order)
tr["is_new_recipient"] = (~tr.duplicated(["initiator", "recipient"])).astype("int8")

# Number of earlier transfers received by the recipient (a simple fan-in measure)
tr["recip_prior_count"] = (
    tr.sort_values(["recipient", "step", "row_id"], kind="stable")  # order by recipient, then time
      .groupby("recipient").cumcount()                              # earlier transfers to the same recipient
)

# Balance-based inputs (leakage ablation set)
for col in ["oldBalInitiator", "newBalInitiator", "oldBalRecipient", "newBalRecipient"]:
    tr[col + "_log"] = np.log1p(tr[col].clip(lower=0))              # log-scale each balance

# 1 when the transfer empties the initiator's account (the simulator's dominant fraud pattern)
tr["drains_account"] = (tr["amount"] == tr["oldBalInitiator"]).astype("int8")

# Amount as a share of the initiator's balance; -1 when the old balance is zero
tr["amount_to_balance"] = (
    tr["amount"] / tr["oldBalInitiator"].where(tr["oldBalInitiator"] > 0)
).fillna(-1)

# Balance consistency errors, compressed with a signed log to limit extreme values
def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))                         # signed log transform

tr["init_bal_diff"] = slog(tr["oldBalInitiator"] - tr["amount"] - tr["newBalInitiator"])
tr["recip_bal_diff"] = slog(tr["newBalRecipient"] - tr["oldBalRecipient"] - tr["amount"])

**Cell 3: Scope, labels, splitting and balancing (steps 6, 7 and 8)**

In [9]:
# The three input sets used for the ablation
FEATS_BEHAV = ["amount_log", "hour", "amt_logratio_own", "amt_z_own", "prior_mean_log",
               "prior_std_log", "steps_since_last", "is_new_recipient", "recip_prior_count"]
FEATS_BAL = ["oldBalInitiator_log", "newBalInitiator_log", "amount_to_balance", "drains_account",
             "init_bal_diff", "oldBalRecipient_log", "newBalRecipient_log", "recip_bal_diff"]
FEATS_TENURE = ["prior_count"]

# Modelling scope: transfers from initiators with at least MIN_HISTORY earlier transfers
MIN_HISTORY = 5
model_df = tr[tr["prior_count"] >= MIN_HISTORY].copy()

# Report what the scope excludes (the cold-start rows that have almost no legitimate examples)
cold = tr[tr["prior_count"] < MIN_HISTORY]
print("Excluded cold-start rows:", len(cold), "| legitimate among them:", (cold["isFraud"] == 0).sum())
print("Modelling scope:\n", model_df["isFraud"].value_counts())

# Step 6: verify the labels and add an evaluation-only stratum flag (never a model input)
assert set(model_df["isFraud"].unique()) <= {0, 1}                  # labels must be binary
model_df["stratum_legit_unusual"] = (
    (model_df["isFraud"] == 0) & (model_df["amt_z_own"] > 3)        # legitimate but unusual for the account
).astype("int8")
print("Legitimate-but-unusual rows:", model_df["stratum_legit_unusual"].sum())

# Step 8: temporal split by step (adjust the cut-offs after checking the printed counts)
TRAIN_END, VAL_END = 115, 153
train = model_df[model_df["step"] <= TRAIN_END]
val = model_df[(model_df["step"] > TRAIN_END) & (model_df["step"] <= VAL_END)]
test = model_df[model_df["step"] > VAL_END]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, "rows:", len(part), "| fraud share:", round(part["isFraud"].mean(), 3),
          "| legit rows:", (part["isFraud"] == 0).sum(),
          "| unusual legit:", part["stratum_legit_unusual"].sum())

# Step 7: class weight for training instead of resampling (fraud is the majority class in this scope)
pos_weight = (train["isFraud"] == 0).sum() / (train["isFraud"] == 1).sum()
print("scale_pos_weight (legit/fraud):", round(pos_weight, 3))

# Realistic-prevalence evaluation set: keep all legitimate rows, down-sample fraud to the target share
def realistic_subset(d, prevalence=0.01, seed=0):
    legit = d[d["isFraud"] == 0]                                    # all legitimate rows are kept
    fraud_all = d[d["isFraud"] == 1]                                # pool of fraud rows
    n_fraud = int(round(len(legit) * prevalence / (1 - prevalence)))  # fraud count that gives the target share
    fraud = fraud_all.sample(n=min(n_fraud, len(fraud_all)), random_state=seed)
    return pd.concat([legit, fraud]).sort_values("step")            # recombine in time order

# Use test only for evaluation; repeat with several seeds and average the metrics later
test_realistic = realistic_subset(test, prevalence=0.01, seed=0)
print("Realistic test set fraud share:", test_realistic["isFraud"].mean())

Excluded cold-start rows: 1663179 | legitimate among them: 3387
Modelling scope:
 isFraud
1    573268
0    274442
Name: count, dtype: int64
Legitimate-but-unusual rows: 3979
train rows: 453460 | fraud share: 0.738 | legit rows: 118763 | unusual legit: 781
val rows: 177977 | fraud share: 0.647 | legit rows: 62793 | unusual legit: 1058
test rows: 216273 | fraud share: 0.571 | legit rows: 92886 | unusual legit: 2140
scale_pos_weight (legit/fraud): 0.355
Realistic test set fraud share: 0.00999744201909959


**Cell 4: Final validation and saving (steps 9 and 10)**

In [10]:
import json
from sklearn.metrics import roc_auc_score

all_feats = FEATS_BEHAV + FEATS_BAL + FEATS_TENURE

# No identifiers, labels or time index may appear among the inputs
banned = ["initiator", "recipient", "row_id", "isFraud", "step", "transactionType"]
assert not any(c in all_feats for c in banned)

# All inputs must be finite (no NaN or infinity)
assert np.isfinite(model_df[all_feats]).all().all()

# The three splits must be strictly ordered in time
assert train["step"].max() < val["step"].min() and val["step"].max() < test["step"].min()

# Leakage scan: AUC of each single feature on a training sample (values near 1.0 indicate leakage)
sample = train.sample(n=min(200_000, len(train)), random_state=0)
rows = []
for f in all_feats:
    auc = roc_auc_score(sample["isFraud"], sample[f])               # AUC of the raw feature value
    rows.append((f, max(auc, 1 - auc)))                             # direction-free strength
scan = pd.DataFrame(rows, columns=["feature", "single_feature_auc"])
print(scan.sort_values("single_feature_auc", ascending=False).to_string(index=False))

# Step 10: save the prepared splits and the feature sets (identifiers are not saved)
OUT = "/kaggle/working"
keep = ["row_id", "step", "isFraud", "stratum_legit_unusual"] + all_feats
train[keep].to_parquet(f"{OUT}/train.parquet", index=False)
val[keep].to_parquet(f"{OUT}/val.parquet", index=False)
test[keep].to_parquet(f"{OUT}/test.parquet", index=False)
test_realistic[keep].to_parquet(f"{OUT}/test_realistic.parquet", index=False)

with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump({"behavioural": FEATS_BEHAV, "balance": FEATS_BAL, "tenure": FEATS_TENURE}, fh, indent=2)
print("Saved to", OUT)

            feature  single_feature_auc
  amount_to_balance            0.919588
oldBalInitiator_log            0.908067
newBalInitiator_log            0.907689
      prior_std_log            0.882325
   amt_logratio_own            0.765475
      init_bal_diff            0.765381
     recip_bal_diff            0.765323
newBalRecipient_log            0.702075
     prior_mean_log            0.674246
          amt_z_own            0.592302
  recip_prior_count            0.590359
        prior_count            0.563713
   is_new_recipient            0.529074
oldBalRecipient_log            0.523718
         amount_log            0.511814
               hour            0.511616
   steps_since_last            0.501590
     drains_account            0.500000
Saved to /kaggle/working


**There is a related risk in the split. Because the split is temporal by row, the same accounts appear in train and test. A model can learn “this account is a fraud account” and score well without learning behaviour. An account-disjoint evaluation would test that, but initiator was not saved to the parquet files. The cell below adds a pseudonymised account code, and it checks all of these points.

Diagnostic cell**

In [11]:
# Remove the constant feature from the balance set and rebuild the full feature list
FEATS_BAL = [f for f in FEATS_BAL if f != "drains_account"]
all_feats = FEATS_BEHAV + FEATS_BAL + FEATS_TENURE

# 1. Absolute-value scan for features whose signal is the size of a deviation, not its sign
#    (reuses the training sample drawn in the previous validation cell)
for f in ["amt_logratio_own", "amt_z_own", "init_bal_diff", "recip_bal_diff"]:
    auc = roc_auc_score(sample["isFraud"], sample[f].abs())   # AUC computed on the magnitude
    print(f, "| AUC on absolute value:", round(max(auc, 1 - auc), 4))

# 2. Share of rows with specific structural patterns, by class
tol = 0.02                                                    # tolerance for rounding in balance arithmetic
flags = pd.DataFrame({
    "initiator balances reconcile": model_df["init_bal_diff"].abs() < tol,        # old - amount = new
    "recipient balances reconcile": model_df["recip_bal_diff"].abs() < tol,       # new - old = amount
    "initiator old balance is zero": model_df["oldBalInitiator_log"] == 0,        # zero or negative old balance
    "recipient old balance is zero": model_df["oldBalRecipient_log"] == 0,        # zero or negative old balance
    "prior std below 0.01 (near-constant amounts)": model_df["prior_std_log"] < 0.01,
})
# Average each flag within each class (columns: legitimate = 0, fraud = 1)
print(flags.groupby(model_df["isFraud"]).mean().T)

# 3. Distribution of the strongest features by class (10th, 50th and 90th percentiles)
cols = ["prior_std_log", "prior_mean_log", "oldBalInitiator_log", "amount_to_balance"]
print(model_df.groupby("isFraud")[cols].quantile([0.1, 0.5, 0.9]).T)

# 4. Pseudonymised integer account code, so account-based splits are possible
#    without storing raw identifiers (this column is never a model input)
model_df["acct_id"] = pd.factorize(model_df["initiator"])[0]

# Rebuild the temporal splits so that they carry the new column
train = model_df[model_df["step"] <= TRAIN_END]
val = model_df[(model_df["step"] > TRAIN_END) & (model_df["step"] <= VAL_END)]
test = model_df[model_df["step"] > VAL_END]

# Share of test rows whose account already appears in the training split
seen = test["acct_id"].isin(train["acct_id"])
print("Share of test rows from accounts seen in training:", round(seen.mean(), 3))

# Row count and fraud rate for seen versus unseen accounts in the test split
print(test.groupby(seen)["isFraud"].agg(["count", "mean"]).rename(index={True: "seen", False: "unseen"}))

# Rebuild the realistic-prevalence test set from the updated test split
test_realistic = realistic_subset(test, prevalence=0.01, seed=0)

# Re-save the prepared files with the account code included
keep = ["row_id", "step", "acct_id", "isFraud", "stratum_legit_unusual"] + all_feats
train[keep].to_parquet(f"{OUT}/train.parquet", index=False)
val[keep].to_parquet(f"{OUT}/val.parquet", index=False)
test[keep].to_parquet(f"{OUT}/test.parquet", index=False)
test_realistic[keep].to_parquet(f"{OUT}/test_realistic.parquet", index=False)

# Re-save the feature sets without the constant feature
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump({"behavioural": FEATS_BEHAV, "balance": FEATS_BAL, "tenure": FEATS_TENURE}, fh, indent=2)

amt_logratio_own | AUC on absolute value: 0.779
amt_z_own | AUC on absolute value: 0.5398
init_bal_diff | AUC on absolute value: 0.7697
recip_bal_diff | AUC on absolute value: 0.7589
isFraud                                              0         1
initiator balances reconcile                  0.978688  0.480247
recipient balances reconcile                  0.978688  0.480247
initiator old balance is zero                 0.020722  0.004092
recipient old balance is zero                 0.552062  0.591503
prior std below 0.01 (near-constant amounts)  0.000171  0.000863
isFraud                      0                               1             \
                           0.1        0.5        0.9       0.1        0.5   
prior_std_log         0.215667   0.438807   0.814768  0.085439   0.117537   
prior_mean_log        9.811284   9.955474  10.709738  9.781642   9.857900   
oldBalInitiator_log  14.733764  18.360001  19.338037  7.961067  10.088795   
amount_to_balance     0.000073   0.000249 

Decisions

Three feature tiers. prior_mean_log, prior_std_log and prior_count are nearly constant per account, so with 99.5% seen accounts they act as account identifiers. They move into a separate profile tier. The headline behavioural set becomes amount_log, hour, amt_logratio_own, amt_z_own, steps_since_last, is_new_recipient and recip_prior_count. The balance tier is unchanged. Ablation runs: behavioural only, plus profile, plus balance, and all combined.
Three evaluation views. (a) The temporal split on all rows, which is the realistic setting where members have history. (b) An account-disjoint split, which shows whether performance depends on memorising accounts. (c) A mixed-account tier, covering the accounts that have both legitimate and fraud rows (about 860 accounts). Here fraud is compared with legitimate transfers from the same accounts, so this is the credible test of per-customer behaviour. Earlier, amount ratio showed almost no separation inside these accounts, so expect this tier to be hard.

In [12]:
# Feature tiers after the leakage diagnostics
FEATS_BEHAV = ["amount_log", "hour", "amt_logratio_own", "amt_z_own",
               "steps_since_last", "is_new_recipient", "recip_prior_count"]   # headline behavioural inputs
FEATS_PROFILE = ["prior_mean_log", "prior_std_log", "prior_count"]            # account-profile ablation tier
# FEATS_BAL is unchanged (drains_account was already removed)
all_feats = FEATS_BEHAV + FEATS_PROFILE + FEATS_BAL

# 1. Inspect the fraud rows whose balances do not reconcile
tol = 0.02                                                    # tolerance for rounding in balance arithmetic
fr = model_df[model_df["isFraud"] == 1]                       # fraud rows only
nonrec = fr[fr["init_bal_diff"].abs() >= tol]                 # fraud rows whose balances do not reconcile
print("Non-reconciling fraud rows:", len(nonrec), "of", len(fr))
# Share of those rows where the balance did not change at all (transfer not applied)
print("Initiator balance unchanged:", (nonrec["newBalInitiator"] == nonrec["oldBalInitiator"]).mean())
print("Recipient balance unchanged:", (nonrec["newBalRecipient"] == nonrec["oldBalRecipient"]).mean())

# 2. Evaluation-only account tiers (these use labels, so they must never be model inputs)
has_legit = (model_df["isFraud"] == 0).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a legitimate transfer
has_fraud = (model_df["isFraud"] == 1).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a fraud transfer
model_df["acct_mixed"] = ((has_legit == 1) & (has_fraud == 1)).astype("int8")                         # account has both
print("Accounts by tier:\n", model_df.groupby("acct_mixed")["acct_id"].nunique())
print("Rows by tier and label:\n", pd.crosstab(model_df["acct_mixed"], model_df["isFraud"]))

# 3. Single-feature AUC inside the mixed-account tier: fraud versus legitimate within the same accounts
mixed = model_df[model_df["acct_mixed"] == 1]                 # rows from mixed accounts only
rows = []
for f in all_feats:
    a_raw = roc_auc_score(mixed["isFraud"], mixed[f])         # AUC on the raw feature value
    a_abs = roc_auc_score(mixed["isFraud"], mixed[f].abs())   # AUC on the magnitude (deviation size)
    rows.append((f, max(a_raw, 1 - a_raw), max(a_abs, 1 - a_abs)))   # direction-free strengths
scan_mixed = pd.DataFrame(rows, columns=["feature", "auc_raw", "auc_abs"])
print(scan_mixed.sort_values("auc_raw", ascending=False).round(3).to_string(index=False))

# 4. Temporal split label for every row (cut-offs defined earlier)
model_df["split_time"] = np.select(
    [model_df["step"] <= TRAIN_END, model_df["step"] <= VAL_END],
    ["train", "val"], default="test")

# 5. Account-disjoint split: 60/20/20 of accounts, shuffled separately within each tier
rng = np.random.default_rng(0)                                # fixed seed for reproducibility
acct_tbl = model_df.groupby("acct_id")["acct_mixed"].first().reset_index()   # one row per account
split_map = {}                                                # account id -> split name
for flag, grp in acct_tbl.groupby("acct_mixed"):
    ids = grp["acct_id"].to_numpy().copy()                    # accounts in this tier
    rng.shuffle(ids)                                          # random order
    n = len(ids)
    n_tr, n_va = int(0.6 * n), int(0.2 * n)                   # train and validation counts
    labels = np.array(["train"] * n_tr + ["val"] * n_va + ["test"] * (n - n_tr - n_va))
    split_map.update(dict(zip(ids, labels)))                  # record the assignment
model_df["split_acct"] = model_df["acct_id"].map(split_map)   # attach the split to every row

# Verify that no account appears in more than one account-disjoint split
sets = {s: set(model_df.loc[model_df["split_acct"] == s, "acct_id"]) for s in ["train", "val", "test"]}
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"])
print(pd.crosstab(model_df["split_acct"], [model_df["acct_mixed"], model_df["isFraud"]]))

# 6. Save one file carrying both split columns, plus the updated feature tiers
OUT = "/kaggle/working"
keep = ["row_id", "step", "acct_id", "isFraud", "stratum_legit_unusual",
        "acct_mixed", "split_time", "split_acct"] + all_feats
model_df[keep].to_parquet(f"{OUT}/model_data.parquet", index=False)
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump({"behavioural": FEATS_BEHAV, "profile": FEATS_PROFILE, "balance": FEATS_BAL}, fh, indent=2)
print("Saved model_data.parquet and feature_sets.json")

Non-reconciling fraud rows: 297958 of 573268
Initiator balance unchanged: 1.0
Recipient balance unchanged: 1.0
Accounts by tier:
 acct_mixed
0    1640
1     804
Name: acct_id, dtype: int64
Rows by tier and label:
 isFraud          0       1
acct_mixed                
0               42  458851
1           274400  114417
            feature  auc_raw  auc_abs
newBalRecipient_log    0.514    0.514
   amt_logratio_own    0.513    0.511
         amount_log    0.513    0.513
          amt_z_own    0.513    0.511
oldBalRecipient_log    0.511    0.511
   is_new_recipient    0.510    0.510
  amount_to_balance    0.510    0.510
  recip_prior_count    0.509    0.509
               hour    0.506    0.506
      prior_std_log    0.504    0.504
     prior_mean_log    0.504    0.504
        prior_count    0.503    0.503
   steps_since_last    0.502    0.502
     recip_bal_diff    0.501    0.506
      init_bal_diff    0.501    0.500
oldBalInitiator_log    0.501    0.501
newBalInitiator_log    0.501    

In [13]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

# Feature tiers for the ablation
tiers = {
    "behavioural": FEATS_BEHAV,                              # transaction-level behaviour only
    "behav+profile": FEATS_BEHAV + FEATS_PROFILE,            # adds account-profile features
    "behav+balance": FEATS_BEHAV + FEATS_BAL,                # adds balance features
    "all": FEATS_BEHAV + FEATS_PROFILE + FEATS_BAL,          # everything
}

def run(data, feats, split_col):
    # Train on the train split, stop early on validation, evaluate on test
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    te_ = data[data[split_col] == "test"]
    model = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=0, verbose=-1)                          # fixed seed for reproducibility
    model.fit(tr_[feats], tr_["isFraud"],
              eval_set=[(va_[feats], va_["isFraud"])], eval_metric="auc",
              callbacks=[lgb.early_stopping(30, verbose=False)])
    p = model.predict_proba(te_[feats])[:, 1]                # fraud probability for each test row
    mixed_te = te_["acct_mixed"].to_numpy() == 1             # mask of test rows from mixed accounts
    return {
        "n_test": len(te_),
        "roc_auc": roc_auc_score(te_["isFraud"], p),                         # ranking quality, all test rows
        "pr_auc": average_precision_score(te_["isFraud"], p),                # precision-recall area
        "auc_mixed_rows": roc_auc_score(te_["isFraud"][mixed_te], p[mixed_te]),  # same, mixed-account rows only
    }

# Scopes: all rows, and the mixed-account tier on its own
scopes = {"all": model_df, "mixed_only": model_df[model_df["acct_mixed"] == 1]}

results = []
for scope_name, data in scopes.items():
    for split_col in ["split_time", "split_acct"]:
        for tier_name, feats in tiers.items():
            r = run(data, feats, split_col)                  # train, evaluate and collect metrics
            r.update({"scope": scope_name, "split": split_col, "tier": tier_name})
            results.append(r)

# Summary table with one row per experiment
res = pd.DataFrame(results)[["scope", "split", "tier", "n_test", "roc_auc", "pr_auc", "auc_mixed_rows"]]
print(res.round(3).to_string(index=False))

     scope      split          tier  n_test  roc_auc  pr_auc  auc_mixed_rows
       all split_time   behavioural  216273    0.840   0.899           0.512
       all split_time behav+profile  216273    0.843   0.902           0.518
       all split_time behav+balance  216273    0.845   0.909           0.513
       all split_time           all  216273    0.848   0.907           0.514
       all split_acct   behavioural  165352    0.877   0.937           0.507
       all split_acct behav+profile  165352    0.886   0.944           0.511
       all split_acct behav+balance  165352    0.895   0.956           0.513
       all split_acct           all  165352    0.896   0.955           0.513
mixed_only split_time   behavioural  131104    0.517   0.299           0.517
mixed_only split_time behav+profile  131104    0.516   0.299           0.516
mixed_only split_time behav+balance  131104    0.516   0.298           0.516
mixed_only split_time           all  131104    0.514   0.297           0.514

In [15]:
# Isolation Forest is used by the unsupervised detector in the evaluation cell
from sklearn.ensemble import IsolationForest

# Settings for the evaluation protocol
PREV = 0.01          # realistic fraud prevalence used for precision and alert-volume estimates
FPRS = [0.01, 0.05]  # fixed false-positive rates at which thresholds are set

def fit_lgb(tr_, va_, feats):
    # Gradient-boosted classifier with early stopping on the validation split
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)
    m.fit(tr_[feats], tr_["isFraud"], eval_set=[(va_[feats], va_["isFraud"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(30, verbose=False)])
    return m

def evaluate(v_s, t_s, va, te, fpr):
    # Threshold chosen on validation legitimate rows so that a share `fpr` of them is flagged
    thr = np.quantile(v_s[va["isFraud"].to_numpy() == 0], 1 - fpr)
    flag = t_s > thr                                            # alerts raised on the test split
    y = te["isFraud"].to_numpy()                                # test labels
    mixed = te["acct_mixed"].to_numpy() == 1                    # rows from accounts with both classes
    unusual = te["stratum_legit_unusual"].to_numpy() == 1       # legitimate-but-unusual rows
    rec = flag[y == 1].mean()                                   # recall over all fraud
    fpr_real = flag[y == 0].mean()                              # realised false-positive rate on test
    alerts = PREV * rec + (1 - PREV) * fpr_real                 # alerts per transaction at prevalence PREV
    return {
        "recall_all": rec,
        "recall_fraud_acct_fraud": flag[(y == 1) & ~mixed].mean(),   # fraud in fraud-dedicated accounts
        "recall_mixed_acct_fraud": flag[(y == 1) & mixed].mean(),    # fraud in ordinary accounts
        "fpr_realised": fpr_real,
        "flag_rate_unusual_legit": flag[unusual].mean(),             # over-flagging of unusual legitimate rows
        "precision_at_1pct": PREV * rec / alerts,                    # expected precision at 1% prevalence
        "alerts_per_1000": 1000 * alerts,                            # expected alerts per 1,000 transactions
    }

rows = []
for split_col in ["split_time", "split_acct"]:
    # Select the training, validation and test rows for this split definition
    tr_ = model_df[model_df[split_col] == "train"]
    va_ = model_df[model_df[split_col] == "val"]
    te_ = model_df[model_df[split_col] == "test"]

    scores = {}  # method name -> (validation scores, test scores); higher means more suspicious

    # Baseline 1: fixed rule, larger amounts are more suspicious
    scores["amount rule"] = (va_["amount_log"].to_numpy(), te_["amount_log"].to_numpy())

    # Baseline 2: deviation rule, larger departure from the account's own norm is more suspicious
    scores["own-norm rule"] = (va_["amt_z_own"].abs().to_numpy(), te_["amt_z_own"].abs().to_numpy())

    # Unsupervised detector fitted on legitimate training rows only (models "normal" behaviour)
    iso = IsolationForest(n_estimators=200, max_samples=50_000, random_state=0, n_jobs=-1)
    iso.fit(tr_.loc[tr_["isFraud"] == 0, FEATS_BEHAV])
    scores["isolation forest"] = (-iso.score_samples(va_[FEATS_BEHAV]),
                                  -iso.score_samples(te_[FEATS_BEHAV]))   # negate: higher = more anomalous

    # Supervised models on two feature tiers
    for name, feats in [("lgbm behavioural", FEATS_BEHAV), ("lgbm all", all_feats)]:
        m = fit_lgb(tr_, va_, feats)
        scores[name] = (m.predict_proba(va_[feats])[:, 1], m.predict_proba(te_[feats])[:, 1])

    # Evaluate every method at every target false-positive rate
    for method, (v_s, t_s) in scores.items():
        auc = roc_auc_score(te_["isFraud"], t_s)                # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)
            r.update({"split": split_col, "method": method, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

# Collect the results into one table and print it
cols = ["split", "method", "fpr_target", "roc_auc", "recall_all", "recall_fraud_acct_fraud",
        "recall_mixed_acct_fraud", "fpr_realised", "flag_rate_unusual_legit",
        "precision_at_1pct", "alerts_per_1000"]
report = pd.DataFrame(rows)[cols]
print(report.round(3).to_string(index=False))

# Drift check: AUC of amount_to_balance on the training split against the test split
for name, part in [("train", train), ("test", test)]:
    a = roc_auc_score(part["isFraud"], part["amount_to_balance"])
    print(name, "amount_to_balance AUC:", round(max(a, 1 - a), 3))

     split           method  fpr_target  roc_auc  recall_all  recall_fraud_acct_fraud  recall_mixed_acct_fraud  fpr_realised  flag_rate_unusual_legit  precision_at_1pct  alerts_per_1000
split_time      amount rule        0.01    0.485       0.000                    0.000                    0.000         0.018                    0.716              0.000           18.023
split_time      amount rule        0.05    0.485       0.030                    0.030                    0.029         0.057                    0.990              0.005           56.754
split_time    own-norm rule        0.01    0.561       0.000                    0.000                    0.000         0.016                    0.683              0.000           15.582
split_time    own-norm rule        0.05    0.561       0.058                    0.072                    0.027         0.055                    1.000              0.010           55.306
split_time isolation forest        0.01    0.508       0.061          

In [16]:
# Tiers redefined so that none can reconstruct the account profile by algebra
FEATS_DEV = ["amt_z_own", "hour", "steps_since_last",
             "is_new_recipient", "recip_prior_count"]        # deviation and context only
tiers2 = {
    "deviation only": FEATS_DEV,                             # strictly behavioural: no amount level, no ratio
    "deviation + amount": FEATS_DEV + ["amount_log"],        # adds transfer size; the account mean stays unrecoverable
    "profile only": FEATS_PROFILE,                           # account-level fingerprint on its own
    "balance only": FEATS_BAL,                               # balance artefacts on their own
    "previous behavioural": FEATS_BEHAV,                     # reference: contains amount_log and amt_logratio_own
}

rows = []
for split_col in ["split_time", "split_acct"]:
    # Select the training, validation and test rows for this split definition
    tr_ = model_df[model_df[split_col] == "train"]
    va_ = model_df[model_df[split_col] == "val"]
    te_ = model_df[model_df[split_col] == "test"]
    for tier, feats in tiers2.items():
        m = fit_lgb(tr_, va_, feats)                         # supervised model on this tier only
        v_s = m.predict_proba(va_[feats])[:, 1]              # validation scores, used to set thresholds
        t_s = m.predict_proba(te_[feats])[:, 1]              # test scores
        auc = roc_auc_score(te_["isFraud"], t_s)             # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)            # recall, FPR and alert volume at this threshold
            r.update({"split": split_col, "tier": tier, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

# One row per split, tier and false-positive target
cols = ["split", "tier", "fpr_target", "roc_auc", "fpr_realised", "recall_all",
        "recall_fraud_acct_fraud", "recall_mixed_acct_fraud",
        "flag_rate_unusual_legit", "alerts_per_1000", "precision_at_1pct"]
print(pd.DataFrame(rows)[cols].round(3).to_string(index=False))

     split                 tier  fpr_target  roc_auc  fpr_realised  recall_all  recall_fraud_acct_fraud  recall_mixed_acct_fraud  flag_rate_unusual_legit  alerts_per_1000  precision_at_1pct
split_time       deviation only        0.01    0.669         0.012       0.071                    0.097                    0.012                    0.007           12.131              0.058
split_time       deviation only        0.05    0.669         0.042       0.179                    0.240                    0.044                    0.011           43.583              0.041
split_time   deviation + amount        0.01    0.783         0.007       0.142                    0.203                    0.007                    0.009            8.575              0.166
split_time   deviation + amount        0.05    0.783         0.042       0.352                    0.492                    0.042                    0.014           44.899              0.078
split_time         profile only        0.01    0.8

In [17]:
# Imports used by this cell (shap is preinstalled on Kaggle; if the import fails, run: pip install shap)
import shap
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Account-disjoint split, so results cannot rely on memorised accounts
tr_ = model_df[model_df["split_acct"] == "train"]
va_ = model_df[model_df["split_acct"] == "val"]
te_ = model_df[model_df["split_acct"] == "test"]

# 1. Drop-one ablation: retrain without each feature and compare test AUC with the full model
base_model = fit_lgb(tr_, va_, FEATS_DEV)                                    # model on all deviation features
base_auc = roc_auc_score(te_["isFraud"], base_model.predict_proba(te_[FEATS_DEV])[:, 1])
print("All deviation features, AUC:", round(base_auc, 3))
for f in FEATS_DEV:
    feats = [x for x in FEATS_DEV if x != f]                                 # feature set with one feature removed
    m = fit_lgb(tr_, va_, feats)                                             # retrain without that feature
    auc = roc_auc_score(te_["isFraud"], m.predict_proba(te_[feats])[:, 1])   # test AUC without it
    print(f"Without {f}: AUC {auc:.3f} (change {auc - base_auc:+.3f})")

# 2. SHAP values on a random sample of test rows (the sample keeps the computation fast)
sample = te_.sample(n=min(20000, len(te_)), random_state=0)
explainer = shap.TreeExplainer(base_model)                                   # exact SHAP values for tree models
sv = explainer.shap_values(sample[FEATS_DEV])
# Different shap versions return a list or a 3-D array for binary models; keep the fraud-class values
if isinstance(sv, list):
    sv = sv[1]
elif sv.ndim == 3:
    sv = sv[:, :, 1]

# Mean absolute SHAP value per feature, ranked
imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATS_DEV).sort_values(ascending=False)
print("Mean |SHAP| per feature:\n", imp.round(4))

# 3. Main drivers behind the five highest-scoring test rows (raw material for alert text)
scores = base_model.predict_proba(sample[FEATS_DEV])[:, 1]                   # fraud score of each sampled row
for i in np.argsort(scores)[::-1][:5]:
    contrib = pd.Series(sv[i], index=FEATS_DEV)                              # contribution of each feature to this row
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(3)   # three largest by magnitude
    print("score", round(float(scores[i]), 3), "| top drivers:", top.round(3).to_dict())

All deviation features, AUC: 0.726
Without amt_z_own: AUC 0.637 (change -0.090)
Without hour: AUC 0.725 (change -0.001)
Without steps_since_last: AUC 0.719 (change -0.007)
Without is_new_recipient: AUC 0.719 (change -0.007)
Without recip_prior_count: AUC 0.674 (change -0.053)
Mean |SHAP| per feature:
 amt_z_own            0.4128
recip_prior_count    0.3687
is_new_recipient     0.1232
steps_since_last     0.0969
hour                 0.0327
dtype: float64
score 0.985 | top drivers: {'amt_z_own': 1.663, 'steps_since_last': 1.233, 'hour': 0.263}
score 0.981 | top drivers: {'steps_since_last': 1.972, 'recip_prior_count': 0.727, 'hour': 0.164}
score 0.98 | top drivers: {'steps_since_last': 1.653, 'recip_prior_count': 0.787, 'amt_z_own': 0.286}
score 0.98 | top drivers: {'steps_since_last': 2.11, 'recip_prior_count': 0.62, 'hour': 0.15}
score 0.979 | top drivers: {'steps_since_last': 1.699, 'recip_prior_count': 0.664, 'amt_z_own': 0.389}


/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [18]:
# Imports used by this cell
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. Fidelity check: base value plus SHAP contributions must reproduce the model's raw log-odds output
ev = np.atleast_1d(explainer.expected_value)[-1]                  # base value for the fraud class
raw = base_model.predict(sample[FEATS_DEV], raw_score=True)       # model output before the sigmoid
print("Max SHAP reconstruction error:", float(np.abs(ev + sv.sum(axis=1) - raw).max()))

# 2. Direction check: fraud rate and mean SHAP value across bins of each feature
shap_df = pd.DataFrame(sv, columns=FEATS_DEV, index=sample.index)  # SHAP values aligned with the sampled rows
for f in FEATS_DEV:
    if sample[f].nunique() <= 2:
        bins = sample[f]                                           # binary feature: group by its two values
    else:
        bins = pd.qcut(sample[f], 5, duplicates="drop")            # otherwise up to five quantile bins
    tab = (pd.DataFrame({"fraud": sample["isFraud"], "shap": shap_df[f]})
             .groupby(bins, observed=True)
             .agg(n=("fraud", "size"), fraud_rate=("fraud", "mean"), mean_shap=("shap", "mean")))
    print(f)
    print(tab.round(3))
    print()

# 3. Alert text built from the SHAP contributions of each row
legit_tr = tr_[tr_["isFraud"] == 0]                                # legitimate training transfers as the peer group
ref = {f: np.sort(legit_tr[f].to_numpy())                          # sorted reference values for percentile lookups
       for f in ["steps_since_last", "recip_prior_count", "amt_z_own"]}

def pctile(f, v):
    # Percentile of value v among legitimate training transfers
    return 100 * np.searchsorted(ref[f], v, side="right") / len(ref[f])

def phrase(f, v):
    # Neutral wording: states the value and its peer percentile, without asserting a direction of risk
    if f == "amt_z_own":
        return (f"the amount deviates from this account's usual transfer size by {v:+.1f} "
                f"(percentile {pctile(f, v):.0f} among legitimate transfers)")
    if f == "recip_prior_count":
        return (f"the recipient had already received {int(v):,} earlier transfers "
                f"(percentile {pctile(f, v):.0f} among legitimate transfers)")
    if f == "steps_since_last":
        return (f"{v:.0f} steps since this account's previous transfer "
                f"(percentile {pctile(f, v):.0f} among legitimate transfers)")
    if f == "is_new_recipient":
        return ("first transfer from this account to this recipient" if v == 1
                else "this account had sent to this recipient before")
    return f"sent at hour {int(v)} of the day"                      # hour feature

scores = base_model.predict_proba(sample[FEATS_DEV])[:, 1]          # fraud score of each sampled row

def make_alert(i, min_contrib=0.1, top_k=3):
    # Cite only features that push the score up, ranked by contribution (log-odds)
    contrib = pd.Series(sv[i], index=FEATS_DEV)                     # SHAP contribution of each feature for row i
    top = contrib[contrib > min_contrib].sort_values(ascending=False).head(top_k)
    if top.empty:
        return f"Risk score {scores[i]:.2f}. No single factor dominates."
    parts = [f"{phrase(f, sample.iloc[i][f])} (+{c:.2f})" for f, c in top.items()]
    return f"Risk score {scores[i]:.2f}. Main factors: " + "; ".join(parts) + "."

# 4. Threshold at a 1% false-positive rate on validation legitimate rows, used to choose alerts to review
v_s = base_model.predict_proba(va_[FEATS_DEV])[:, 1]                # validation scores
thr = np.quantile(v_s[va_["isFraud"].to_numpy() == 0], 0.99)        # score cut-off for alerts
flagged = np.where(scores > thr)[0]                                 # sampled rows that would raise an alert
rng = np.random.default_rng(1)                                      # fixed seed for a reproducible selection
top5 = np.argsort(scores)[::-1][:5]                                 # five highest-scoring rows
legit_alerts = [i for i in flagged if sample.iloc[i]["isFraud"] == 0][:3]   # first three false alarms
rand_alerts = rng.choice(flagged, size=min(3, len(flagged)), replace=False) # three random alerts
for i in list(top5) + legit_alerts + list(rand_alerts):
    # True label and account tier are for review only and never appear in the alert text
    print(f"[true label {int(sample.iloc[i]['isFraud'])}, mixed account {int(sample.iloc[i]['acct_mixed'])}]",
          make_alert(i))

# 5. Does amt_z_own carry deviation or account regularity? Swap it for the raw log-ratio,
#    which has no account-specific denominator
feats_alt = ["amt_logratio_own" if f == "amt_z_own" else f for f in FEATS_DEV]   # same set with the swap
m_alt = fit_lgb(tr_, va_, feats_alt)                                # retrain with the swapped feature
auc_alt = roc_auc_score(te_["isFraud"], m_alt.predict_proba(te_[feats_alt])[:, 1])
print("AUC with amt_logratio_own instead of amt_z_own:", round(auc_alt, 3), "| original:", round(base_auc, 3))

Max SHAP reconstruction error: 1.6542323066914832e-14
amt_z_own
                      n  fraud_rate  mean_shap
amt_z_own                                     
(-3.899, -0.248]   4000       0.739      0.038
(-0.248, -0.207]   4000       0.644     -0.150
(-0.207, -0.161]   4000       0.548     -0.404
(-0.161, -0.0165]  4000       0.635     -0.027
(-0.0165, 28.339]  4000       0.742      0.393

hour
                  n  fraud_rate  mean_shap
hour                                      
(-0.001, 4.0]  4036       0.666      0.024
(4.0, 10.0]    4713       0.669      0.039
(10.0, 15.0]   4093       0.675      0.004
(15.0, 19.0]   3484       0.650     -0.019
(19.0, 23.0]   3674       0.643     -0.045

steps_since_last
                      n  fraud_rate  mean_shap
steps_since_last                              
(-0.001, 1.0]     18907       0.655     -0.038
(1.0, 86.0]        1093       0.771      0.761

is_new_recipient
                      n  fraud_rate  mean_shap
is_new_recipient             

In [19]:
# Imports used by this cell
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# 1. Time-proxy check for recip_prior_count
chk = model_df.assign(
    cnt_bin=pd.qcut(model_df["recip_prior_count"], 4, duplicates="drop"),   # four count bins with equal row share
    step_bin=pd.cut(model_df["step"], [-1, 48, 96, 144, 192]))              # four time ranges across steps 0-192

# Median step per count bin: very different medians mean the count mostly tracks time
print(chk.groupby("cnt_bin", observed=True)["step"].median())

# Fraud rate by count bin within each time range: a genuine count effect shows down each column
print(chk.pivot_table(index="cnt_bin", columns="step_bin", values="isFraud",
                      aggfunc="mean", observed=True).round(3))

# Row counts per cell, to judge how reliable each rate is
print(chk.pivot_table(index="cnt_bin", columns="step_bin", values="isFraud",
                      aggfunc="size", observed=True))

# 2. Windowed recipient counts: transfers received in the previous 6 and 24 steps (current step excluded)
WINDOWS = [6, 24]                                    # window lengths in steps
n_steps = int(tr["step"].max()) + 1                  # number of distinct step values (0 to max)

# Transfers received per recipient per step (rows: recipients, columns: steps)
cnt = tr.groupby(["recipient", "step"]).size().unstack(fill_value=0)
cnt = cnt.reindex(columns=range(n_steps), fill_value=0)       # make sure every step has a column

# cs[:, k] holds the transfers received in steps before k (a leading zero column makes the lookup simple)
cs = np.concatenate([np.zeros((len(cnt), 1)), cnt.to_numpy().cumsum(axis=1)], axis=1)

rec_idx = pd.Index(cnt.index).get_indexer(tr["recipient"])    # matrix row of each transfer's recipient
st = tr["step"].to_numpy()                                    # step of each transfer
for w in WINDOWS:
    lo = np.maximum(st - w, 0)                                # window start (rows before step w see a shorter window)
    tr[f"recip_cnt_{w}"] = cs[rec_idx, st] - cs[rec_idx, lo]  # transfers received in steps [st - w, st - 1]

# Attach the new columns to the modelling table through row_id (drop old copies if the cell is re-run)
new_cols = [f"recip_cnt_{w}" for w in WINDOWS]
model_df = model_df.drop(columns=[c for c in new_cols if c in model_df.columns])
model_df = model_df.merge(tr[["row_id"] + new_cols], on="row_id", how="left")
print("Missing values in new features:", int(model_df[new_cols].isna().sum().sum()))

# 3. Re-run the strict tier with each recipient-count variant
tiers3 = {
    "cumulative count (reference)": FEATS_DEV,
    "windowed counts": ["amt_z_own", "hour", "steps_since_last", "is_new_recipient"] + new_cols,
    "no recipient counts": ["amt_z_own", "hour", "steps_since_last", "is_new_recipient"],
}

rows = []
for split_col in ["split_time", "split_acct"]:
    # Training, validation and test rows for this split definition
    tr_ = model_df[model_df[split_col] == "train"]
    va_ = model_df[model_df[split_col] == "val"]
    te_ = model_df[model_df[split_col] == "test"]
    for tier, feats in tiers3.items():
        m = fit_lgb(tr_, va_, feats)                          # supervised model on this tier
        v_s = m.predict_proba(va_[feats])[:, 1]               # validation scores, used to set thresholds
        t_s = m.predict_proba(te_[feats])[:, 1]               # test scores
        auc = roc_auc_score(te_["isFraud"], t_s)              # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)             # recall, FPR and alert volume at this threshold
            r.update({"split": split_col, "tier": tier, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

cols = ["split", "tier", "fpr_target", "roc_auc", "fpr_realised", "recall_all",
        "recall_mixed_acct_fraud", "flag_rate_unusual_legit", "alerts_per_1000"]
print(pd.DataFrame(rows)[cols].round(3).to_string(index=False))

cnt_bin
(-0.001, 22.0]     37.0
(22.0, 45.0]       85.0
(45.0, 67.0]      134.0
(67.0, 192.0]     174.0
Name: step, dtype: float64
step_bin        (-1, 48]  (48, 96]  (96, 144]  (144, 192]
cnt_bin                                                  
(-0.001, 22.0]     0.830     0.700      0.003       0.000
(22.0, 45.0]       0.566     0.696      0.680       0.179
(45.0, 67.0]       0.019     0.682      0.668       0.614
(67.0, 192.0]      0.000     0.174      0.632       0.576
step_bin        (-1, 48]  (48, 96]  (96, 144]  (144, 192]
cnt_bin                                                  
(-0.001, 22.0]    151369     58144       1207        2370
(22.0, 45.0]         851    155067      62550         123
(45.0, 67.0]          54      2585     148034       58236
(67.0, 192.0]          1        23       6415      200681
Missing values in new features: 0
     split                         tier  fpr_target  roc_auc  fpr_realised  recall_all  recall_mixed_acct_fraud  flag_rate_unusual_legit  a

In [20]:
import json
import numpy as np
import pandas as pd

# Final feature tiers; the cumulative recipient count is excluded as a time proxy
FEATS_STRICT = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient",
                "recip_cnt_6", "recip_cnt_24"]                    # strict behavioural tier
FEATS_AMOUNT = FEATS_STRICT + ["amount_log"]                      # strict tier plus transfer size
tiers_final = {
    "strict_behavioural": FEATS_STRICT,
    "behavioural_plus_amount": FEATS_AMOUNT,
    "profile": FEATS_PROFILE,
    "balance": FEATS_BAL,
}
feature_cols = sorted({f for fs in tiers_final.values() for f in fs})   # every input column, no duplicates

# Columns used for identification, splitting and evaluation only (never model inputs)
meta_cols = ["row_id", "step", "acct_id", "isFraud", "stratum_legit_unusual",
             "acct_mixed", "split_time", "split_acct"]

# Validation checks before saving
assert not set(feature_cols) & set(meta_cols + ["initiator", "recipient"])   # no labels or identifiers among the inputs
assert model_df[feature_cols].notna().all().all()                            # no missing values
assert np.isfinite(model_df[feature_cols].to_numpy()).all()                  # no infinite values
assert model_df["isFraud"].isin([0, 1]).all()                                # binary labels
assert model_df["row_id"].is_unique                                          # every row appears once

# The temporal split must be strictly ordered in time
t = model_df.groupby("split_time")["step"].agg(["min", "max"])
assert t.loc["train", "max"] < t.loc["val", "min"] and t.loc["val", "max"] < t.loc["test", "min"]

# The account split must not share any account between parts
a = model_df.groupby("split_acct")["acct_id"].agg(set)
assert not (a["train"] & a["val"]) and not (a["train"] & a["test"]) and not (a["val"] & a["test"])

# Summary of the saved dataset: size and class counts per split definition
print(model_df.shape)
print(pd.crosstab(model_df["split_time"], model_df["isFraud"]))
print(pd.crosstab(model_df["split_acct"], model_df["isFraud"]))

# Save the dataset and the tier definitions
OUT = "/kaggle/working"
model_df[meta_cols + feature_cols].to_parquet(f"{OUT}/model_data.parquet", index=False)
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump(tiers_final, fh, indent=2)
print("Saved model_data.parquet and feature_sets.json")

(847710, 37)
isFraud          0       1
split_time                
test         92886  123387
train       118763  334697
val          62793  115184
isFraud          0       1
split_acct                
test         56149  109203
train       171086  345428
val          47207  118637
Saved model_data.parquet and feature_sets.json


Cell: final training and evaluation from the saved files

This cell is self-contained, so it also works after a session restart. It evaluates every final tier on both splits, adds the three baselines, reports account-resampled intervals for the main metrics, and saves the results and the strict-tier models for the SHAP step.

In [21]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

# Load the prepared dataset and the tier definitions saved earlier
OUT = "/kaggle/working"
data = pd.read_parquet(f"{OUT}/model_data.parquet")
tiers = json.load(open(f"{OUT}/feature_sets.json"))
print("Reloaded shape:", data.shape)                      # expect 25 columns: 8 metadata + 17 inputs

PREV = 0.01              # realistic fraud prevalence for precision and alert volume
FPRS = [0.01, 0.05]      # target false-positive rates for threshold setting
N_BOOT = 200             # account-level bootstrap resamples

def fit_lgb(tr_, va_, feats):
    # Gradient-boosted classifier with early stopping on the validation split
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)
    m.fit(tr_[feats], tr_["isFraud"], eval_set=[(va_[feats], va_["isFraud"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(30, verbose=False)])
    return m

def metrics(flag, y, mixed, unusual):
    # Metrics for one set of alert flags on one set of test rows
    rec = flag[y == 1].mean()                                    # recall over all fraud
    fpr = flag[y == 0].mean()                                    # false-positive rate over legitimate rows
    alerts = PREV * rec + (1 - PREV) * fpr                       # alerts per transaction at prevalence PREV
    return {
        "recall_all": rec,
        "recall_fraud_acct_fraud": flag[(y == 1) & ~mixed].mean(),   # fraud in fraud-dedicated accounts
        "recall_mixed_acct_fraud": flag[(y == 1) & mixed].mean(),    # fraud in ordinary accounts
        "fpr": fpr,
        "flag_rate_unusual_legit": flag[unusual].mean() if unusual.any() else np.nan,
        "precision_at_1pct": PREV * rec / alerts if alerts > 0 else np.nan,
        "alerts_per_1000": 1000 * alerts,
    }

def evaluate(v_s, t_s, va, te, fpr_target):
    # Threshold set on validation legitimate rows, then applied to the test scores
    thr = np.quantile(v_s[va["isFraud"].to_numpy() == 0], 1 - fpr_target)
    flag = t_s > thr
    y = te["isFraud"].to_numpy()
    mixed = te["acct_mixed"].to_numpy() == 1
    unusual = te["stratum_legit_unusual"].to_numpy() == 1
    out = metrics(flag, y, mixed, unusual)

    # Account-level bootstrap: resample test accounts with replacement and recompute two metrics
    acc = te["acct_id"].to_numpy()
    idx_by_acct = pd.Series(np.arange(len(te))).groupby(acc).apply(np.asarray)   # row positions per account
    rng = np.random.default_rng(0)
    boots = []
    for _ in range(N_BOOT):
        pick = rng.choice(len(idx_by_acct), size=len(idx_by_acct), replace=True) # resample accounts
        rows = np.concatenate(idx_by_acct.iloc[pick].to_numpy())                 # all rows of the picked accounts
        b = metrics(flag[rows], y[rows], mixed[rows], unusual[rows])
        boots.append((b["recall_all"], b["fpr"]))
    boots = np.array(boots)
    out["recall_all_lo"], out["recall_all_hi"] = np.nanpercentile(boots[:, 0], [2.5, 97.5])
    out["fpr_lo"], out["fpr_hi"] = np.nanpercentile(boots[:, 1], [2.5, 97.5])
    return out

rows = []
for split_col in ["split_time", "split_acct"]:
    # Training, validation and test rows for this split definition
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    te_ = data[data[split_col] == "test"]

    scores = {}   # method name -> (validation scores, test scores); higher means more suspicious

    # Baselines: fixed amount rule, own-norm deviation rule, Isolation Forest on the strict tier
    scores["rule: amount"] = (va_["amount_log"].to_numpy(), te_["amount_log"].to_numpy())
    scores["rule: own-norm"] = (va_["amt_z_own"].abs().to_numpy(), te_["amt_z_own"].abs().to_numpy())
    iso = IsolationForest(n_estimators=200, max_samples=50_000, random_state=0, n_jobs=-1)
    iso.fit(tr_.loc[tr_["isFraud"] == 0, tiers["strict_behavioural"]])    # fitted on legitimate rows only
    scores["isolation forest"] = (-iso.score_samples(va_[tiers["strict_behavioural"]]),
                                  -iso.score_samples(te_[tiers["strict_behavioural"]]))

    # Supervised models, one per final tier; the strict-tier model is kept for the SHAP step
    for name, feats in tiers.items():
        m = fit_lgb(tr_, va_, feats)
        scores[f"lgbm: {name}"] = (m.predict_proba(va_[feats])[:, 1], m.predict_proba(te_[feats])[:, 1])
        if name == "strict_behavioural":
            joblib.dump(m, f"{OUT}/model_strict_{split_col}.joblib")      # saved for the explanation layer

    # Evaluate every method at every target false-positive rate
    for method, (v_s, t_s) in scores.items():
        auc = roc_auc_score(te_["isFraud"], t_s)                          # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)
            r.update({"split": split_col, "method": method, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

# Collect, print and save the results table
cols = ["split", "method", "fpr_target", "roc_auc", "fpr", "fpr_lo", "fpr_hi",
        "recall_all", "recall_all_lo", "recall_all_hi",
        "recall_fraud_acct_fraud", "recall_mixed_acct_fraud",
        "flag_rate_unusual_legit", "alerts_per_1000", "precision_at_1pct"]
res = pd.DataFrame(rows)[cols]
res.to_csv(f"{OUT}/final_results.csv", index=False)
print(res.round(3).to_string(index=False))

Reloaded shape: (847710, 25)
     split                        method  fpr_target  roc_auc   fpr  fpr_lo  fpr_hi  recall_all  recall_all_lo  recall_all_hi  recall_fraud_acct_fraud  recall_mixed_acct_fraud  flag_rate_unusual_legit  alerts_per_1000  precision_at_1pct
split_time                  rule: amount        0.01    0.485 0.018   0.017   0.020       0.000          0.000          0.000                    0.000                    0.000                    0.716           18.023              0.000
split_time                  rule: amount        0.05    0.485 0.057   0.046   0.070       0.030          0.021          0.038                    0.030                    0.029                    0.990           56.754              0.005
split_time                rule: own-norm        0.01    0.561 0.016   0.015   0.017       0.000          0.000          0.000                    0.000                    0.000                    0.683           15.582              0.000
split_time             

Strict tier against the baselines, nominal 1% false-positive target

Each cell shows realised FPR and recall. FPR is measured on test, so compare methods at their realised FPR and not at the nominal target.

Method	Time split	Account split
Amount rule	1.8% → 0.0%	0.9% → 0.0%
Own-norm rule	1.6% → 0.0%	1.0% → 0.1%
Isolation Forest	1.2% → 0.7%	0.9% → 0.9%
Strict behavioural model	0.8% → 4.2%	1.3% → 8.6%
Behavioural + amount	0.6% → 11.2%	1.7% → 18.5%

What the intervals support

The strict model beats all three baselines. The recall intervals do not overlap: 4.0% to 4.3% against 0.6% to 0.7% for the best baseline on the time split, and 8.1% to 9.2% against 0.9% to 1.0% on the account split. At the 5% target the strict model reaches 7.7% recall at 2.7% FPR (time) and 19.9% at 5.0% (account), while the own-norm rule reaches 5.8% at 5.5% and 5.7% at 4.6%. The strict model’s FPR intervals are wide (0.8% to 2.1% on the account split) and overlap the baselines’ at the 1% target. The recall gap is far larger than that difference could explain.
Mixed-account recall equals the false-positive rate for every model tier. Across all 16 rows for the four supervised tiers, the two figures differ by at most 0.2 percentage points (for example 1.2% against 1.3%, and 5.0% against 5.0%, for the strict tier on the account split). The table gives no interval for mixed-account recall, so this is a point comparison, but the FPR intervals are many times wider than the gap. Within ordinary accounts, no tier separates fraud from legitimate transfers.
The rules and Isolation Forest do worse than chance inside ordinary accounts. Own-norm rule: 2.7% recall against 5.5% FPR (time split). Amount rule: 1.3% against 3.7% (account split). They flag legitimate transfers more often than fraud, because large deviations in this data are legitimate.
Flagging of unusual legitimate transfers. The strict tier flags 0.9% to 3.0% of them, near its overall FPR. The amount rule flags 46% to 99% and Isolation Forest 32% to 100%. The own-norm rule’s 68% to 100% is circular and should not be reported as evidence. The low model figure reflects the simulator’s direction, as noted earlier.
Alert volume. At 1% prevalence the strict tier raises 7.9 (time) and 13.8 (account) alerts per 1,000 in-scope transfers, with precision of 5.2% and 6.3%. About one alert in 16 to 19 would be genuine. The profile and balance tiers reach 20% to 58% precision, but they rely on the account fingerprint and the unapplied-transfer artefact.

Limits on the intervals

They cover test sampling only. The bootstrap resamples test accounts, but the model and the validation-based threshold are fixed, so training randomness and threshold variation are not included.
Thresholds did not always transfer. The 5% target gave a realised FPR of 2.7% (2.5% to 2.9%) for the strict model on the time split, and 8.8% (8.5% to 9.0%) for Isolation Forest. Neither interval contains the target. The profile tier at the 1% account-split target has an FPR interval of 0.1% to 2.9%, because validation and test hold legitimate rows from only about 160 accounts each.
Report both views. For the final report, put recall at matched FPR next to the nominal-target table. I can add that cell if you want it.

In [2]:
# Imports used by the rebuild
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

# Paths: the raw file is read-only input, results are written to the working directory
RAW = "/kaggle/input/datasets/linzkibet/frauddetectiodata/Synthetic Mobile Money Transaction Dataset/MoMTSim_20240722202413_1000_dataset.csv"
OUT = "/kaggle/working"

# Settings carried over from the earlier cells
MIN_HISTORY = 5            # minimum earlier transfers for a row to enter the modelling scope
TRAIN_END, VAL_END = 115, 153   # temporal split cut-offs (steps)
Z_FLOOR = 0.1              # floor added to the standard deviation in the z-score
WINDOWS = [6, 24]          # windows (steps) for the recipient counts

# 1. Load the raw file and keep TRANSFER rows, the only type that contains fraud
df = pd.read_csv(RAW)
tr = df[df["transactionType"] == "TRANSFER"].copy()
tr = tr.rename_axis("row_id").reset_index()          # row_id = position in the raw file
del df                                               # free memory
tr["initiator"] = tr["initiator"].astype(str)        # identifiers stored as strings
tr["recipient"] = tr["recipient"].astype(str)

# 2. Cleaning: drop exact duplicates and non-positive amounts (outliers are kept)
dup_mask = tr.drop(columns="row_id").duplicated()
tr = tr[~dup_mask & (tr["amount"] > 0)].reset_index(drop=True)
print("Rows after cleaning:", len(tr))               # expect 2,510,889

# 3. Transformations
tr["amount_log"] = np.log1p(tr["amount"])            # log-scaled amount
tr["amount_log_sq"] = tr["amount_log"] ** 2          # helper column for the running variance
tr["hour"] = tr["step"] % 24                         # hour of day, assuming one step is one hour

# Sort by initiator then time, so every running statistic sees only earlier transfers
tr = tr.sort_values(["initiator", "step", "row_id"], kind="stable").reset_index(drop=True)
g = tr.groupby("initiator")                          # one group per initiating account

# 4. Account-history features, computed from earlier transfers only
tr["prior_count"] = g.cumcount()                                   # earlier transfers by this account
n = tr["prior_count"].where(tr["prior_count"] > 0)                 # same count, NaN when there is no history
s1 = g["amount_log"].cumsum() - tr["amount_log"]                   # sum of earlier log amounts
s2 = g["amount_log_sq"].cumsum() - tr["amount_log_sq"]             # sum of earlier squared log amounts
tr["prior_mean_log"] = s1 / n                                      # account's typical amount so far
var = (s2 / n) - tr["prior_mean_log"] ** 2                         # running variance
tr["prior_std_log"] = np.sqrt(var.clip(lower=0))                   # clip guards against tiny negative values
tr["amt_logratio_own"] = tr["amount_log"] - tr["prior_mean_log"]   # helper for the z-score
tr["amt_z_own"] = tr["amt_logratio_own"] / (tr["prior_std_log"] + Z_FLOOR)   # deviation from the account's norm
tr["steps_since_last"] = g["step"].diff()                          # steps since the account's previous transfer
tr["is_new_recipient"] = (~tr.duplicated(["initiator", "recipient"])).astype("int8")   # first send to this recipient

# 5. Balance-based features (the artefact tier)
for col in ["oldBalInitiator", "newBalInitiator", "oldBalRecipient", "newBalRecipient"]:
    tr[col + "_log"] = np.log1p(tr[col].clip(lower=0))             # log-scaled balances, negatives clipped to 0
tr["amount_to_balance"] = (tr["amount"] / tr["oldBalInitiator"].where(tr["oldBalInitiator"] > 0)).fillna(-1)

def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))                        # signed log transform

tr["init_bal_diff"] = slog(tr["oldBalInitiator"] - tr["amount"] - tr["newBalInitiator"])    # initiator balance error
tr["recip_bal_diff"] = slog(tr["newBalRecipient"] - tr["oldBalRecipient"] - tr["amount"])   # recipient balance error

# 6. Windowed recipient counts: transfers received in the previous 6 and 24 steps (current step excluded)
n_steps = int(tr["step"].max()) + 1                                # number of step values (0 to max)
cnt = tr.groupby(["recipient", "step"]).size().unstack(fill_value=0)   # recipients x steps matrix of counts
cnt = cnt.reindex(columns=range(n_steps), fill_value=0)            # make sure every step has a column
cs = np.concatenate([np.zeros((len(cnt), 1)), cnt.to_numpy().cumsum(axis=1)], axis=1)   # cs[:, k] = received before step k
rec_idx = pd.Index(cnt.index).get_indexer(tr["recipient"])         # matrix row of each transfer's recipient
st = tr["step"].to_numpy()                                         # step of each transfer
for w in WINDOWS:
    lo = np.maximum(st - w, 0)                                     # window start
    tr[f"recip_cnt_{w}"] = cs[rec_idx, st] - cs[rec_idx, lo]       # transfers received in steps [st - w, st - 1]
del cnt, cs                                                        # free memory

# 7. Modelling scope, evaluation-only flags and account identifier
model_df = tr[tr["prior_count"] >= MIN_HISTORY].copy()             # history-rich transfers only
model_df["stratum_legit_unusual"] = ((model_df["isFraud"] == 0) & (model_df["amt_z_own"] > 3)).astype("int8")
model_df["acct_id"] = pd.factorize(model_df["initiator"])[0]       # pseudonymised integer account code
has_legit = (model_df["isFraud"] == 0).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a legitimate transfer
has_fraud = (model_df["isFraud"] == 1).astype("int8").groupby(model_df["acct_id"]).transform("max")   # account has a fraud transfer
model_df["acct_mixed"] = ((has_legit == 1) & (has_fraud == 1)).astype("int8")   # account has both (evaluation only)

# 8. Temporal split by step
model_df["split_time"] = np.select(
    [model_df["step"] <= TRAIN_END, model_df["step"] <= VAL_END],
    ["train", "val"], default="test")

# 9. Account-disjoint split: 60/20/20 of accounts, shuffled separately within each tier
rng = np.random.default_rng(0)                                     # fixed seed, same as before
acct_tbl = model_df.groupby("acct_id")["acct_mixed"].first().reset_index()   # one row per account
split_map = {}                                                     # account id -> split name
for flag, grp in acct_tbl.groupby("acct_mixed"):
    ids = grp["acct_id"].to_numpy().copy()                         # accounts in this tier
    rng.shuffle(ids)                                               # random order
    n_acc = len(ids)
    n_tr, n_va = int(0.6 * n_acc), int(0.2 * n_acc)                # train and validation counts
    labels = np.array(["train"] * n_tr + ["val"] * n_va + ["test"] * (n_acc - n_tr - n_va))
    split_map.update(dict(zip(ids, labels)))                       # record the assignment
model_df["split_acct"] = model_df["acct_id"].map(split_map)        # attach the split to every row

# 10. Compare the rebuilt data with the counts printed earlier (rows: legitimate, fraud)
expected_time = {"train": (118763, 334697), "val": (62793, 115184), "test": (92886, 123387)}
expected_acct = {"train": (171086, 345428), "val": (47207, 118637), "test": (56149, 109203)}
for col, expected in [("split_time", expected_time), ("split_acct", expected_acct)]:
    ct = pd.crosstab(model_df[col], model_df["isFraud"])           # class counts per split
    got = {s: (int(ct.loc[s, 0]), int(ct.loc[s, 1])) for s in ct.index}
    print(col, "matches earlier run:", got == expected)
    print(ct)
assert len(model_df) == 847710                                     # modelling scope size from the earlier run

# 11. Tier definitions and final validation checks
FEATS_STRICT = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient", "recip_cnt_6", "recip_cnt_24"]
FEATS_PROFILE = ["prior_mean_log", "prior_std_log", "prior_count"]
FEATS_BAL = ["oldBalInitiator_log", "newBalInitiator_log", "amount_to_balance", "init_bal_diff",
             "oldBalRecipient_log", "newBalRecipient_log", "recip_bal_diff"]
tiers_final = {
    "strict_behavioural": FEATS_STRICT,
    "behavioural_plus_amount": FEATS_STRICT + ["amount_log"],
    "profile": FEATS_PROFILE,
    "balance": FEATS_BAL,
}
feature_cols = sorted({f for fs in tiers_final.values() for f in fs})        # every input column, no duplicates
meta_cols = ["row_id", "step", "acct_id", "isFraud", "stratum_legit_unusual",
             "acct_mixed", "split_time", "split_acct"]                        # identification and evaluation columns
assert model_df[feature_cols].notna().all().all()                            # no missing values
assert np.isfinite(model_df[feature_cols].to_numpy()).all()                  # no infinite values
assert model_df["row_id"].is_unique                                          # every row appears once
a = model_df.groupby("split_acct")["acct_id"].agg(set)                       # accounts per split
assert not (a["train"] & a["val"]) and not (a["train"] & a["test"]) and not (a["val"] & a["test"])

# 12. Save the dataset and the tier definitions
model_df[meta_cols + feature_cols].to_parquet(f"{OUT}/model_data.parquet", index=False)
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump(tiers_final, fh, indent=2)

# 13. Retrain and save the strict-tier model for each split (needed by the SHAP cell)
data = model_df[meta_cols + feature_cols]                                    # the saved columns only
for split_col in ["split_time", "split_acct"]:
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)                       # same settings as the earlier runs
    m.fit(tr_[FEATS_STRICT], tr_["isFraud"],
          eval_set=[(va_[FEATS_STRICT], va_["isFraud"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(30, verbose=False)])                 # early stopping on the validation split
    joblib.dump(m, f"{OUT}/model_strict_{split_col}.joblib")                 # saved for the explanation layer
print("Saved model_data.parquet, feature_sets.json and both strict-tier models to", OUT)

Rows after cleaning: 2510889
split_time matches earlier run: True
isFraud          0       1
split_time                
test         92886  123387
train       118763  334697
val          62793  115184
split_acct matches earlier run: True
isFraud          0       1
split_acct                
test         56149  109203
train       171086  345428
val          47207  118637
Saved model_data.parquet, feature_sets.json and both strict-tier models to /kaggle/working


In [3]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import shap

OUT = "/kaggle/working"
SPLIT = "split_acct"      # split whose strict-tier model is explained ("split_time" is also available)

# Load the prepared data, the tier definitions and the saved strict-tier model
data = pd.read_parquet(f"{OUT}/model_data.parquet")
tiers = json.load(open(f"{OUT}/feature_sets.json"))
FEATS = tiers["strict_behavioural"]
model = joblib.load(f"{OUT}/model_strict_{SPLIT}.joblib")

# Train, validation and test rows for the chosen split
tr_ = data[data[SPLIT] == "train"]
va_ = data[data[SPLIT] == "val"]
te_ = data[data[SPLIT] == "test"]

# 1. SHAP values on a random sample of test rows
sample = te_.sample(n=min(20000, len(te_)), random_state=0).reset_index(drop=True)
explainer = shap.TreeExplainer(model)                          # exact SHAP values for tree models
sv = explainer.shap_values(sample[FEATS])
if isinstance(sv, list):                                       # some shap versions return one array per class
    sv = sv[1]
elif sv.ndim == 3:                                             # others return a 3-D array
    sv = sv[:, :, 1]

# Fidelity check: base value plus contributions must reproduce the model's raw log-odds
ev = np.atleast_1d(explainer.expected_value)[-1]
raw = model.predict(sample[FEATS], raw_score=True)
print("Max SHAP reconstruction error:", float(np.abs(ev + sv.sum(axis=1) - raw).max()))

# Mean absolute SHAP value per feature, ranked
imp = pd.Series(np.abs(sv).mean(axis=0), index=FEATS).sort_values(ascending=False)
print("Mean |SHAP| per feature:\n", imp.round(4), "\n")

# 2. Direction tables: fraud rate and mean SHAP value across bins of each feature
shap_df = pd.DataFrame(sv, columns=FEATS)                      # SHAP values aligned with the sampled rows
for f in FEATS:
    if sample[f].nunique() <= 2:
        bins = sample[f]                                       # binary feature: group by its two values
    else:
        bins = pd.qcut(sample[f], 8, duplicates="drop")        # otherwise up to eight quantile bins
    tab = (pd.DataFrame({"fraud": sample["isFraud"], "shap": shap_df[f]})
             .groupby(bins, observed=True)
             .agg(n=("fraud", "size"), fraud_rate=("fraud", "mean"), mean_shap=("shap", "mean")))
    print(f)
    print(tab.round(3))
    print()

# 3. Reference distributions: legitimate training transfers, used to place a value among its peers
legit_ref = tr_[tr_["isFraud"] == 0]
ref = {f: np.sort(legit_ref[f].to_numpy()) for f in FEATS}     # sorted values for fast percentile lookups
TAIL = 0.05                                                    # a value is "extreme" if at most 5% of legitimate rows are as extreme
CITABLE = [f for f in FEATS if f != "hour"]                    # hour is never cited (no signal in this data)

def tail_info(f, v):
    # Returns which tail the value sits in and the share of legitimate rows at least as extreme
    arr = ref[f]
    n = len(arr)
    upper = 1 - np.searchsorted(arr, v, side="left") / n       # share of legitimate rows with value >= v
    lower = np.searchsorted(arr, v, side="right") / n          # share of legitimate rows with value <= v
    return ("high", upper) if upper <= lower else ("low", lower)

def fmt(f, v, side, share):
    # Wording states the value and its position among legitimate transfers, with no claim about risk direction
    pos = f"{'above' if side == 'high' else 'below'} {100 * (1 - share):.0f}% of legitimate transfers"
    if f == "amt_z_own":
        return f"amount deviation from this account's usual transfer size (z-score {v:+.1f}, {pos})"
    if f == "steps_since_last":
        return f"{v:.0f} steps since this account's previous transfer ({pos})"
    if f == "is_new_recipient":
        return ("first transfer from this account to this recipient" if v == 1
                else "this account had sent to this recipient before")
    if f == "recip_cnt_6":
        return f"recipient received {int(v)} transfers in the previous 6 steps ({pos})"
    return f"recipient received {int(v)} transfers in the previous 24 steps ({pos})"   # recip_cnt_24

def factors(i, min_contrib=0.1, top_k=3):
    # Citable factors for row i: positive SHAP contribution and a value in the outer tail
    out = []
    for f in CITABLE:
        c = sv[i, FEATS.index(f)]                              # contribution of this feature to row i (log-odds)
        if c <= min_contrib:
            continue
        side, share = tail_info(f, sample.at[i, f])
        if share > TAIL:
            continue
        out.append((c, f, side, share))
    return sorted(out, reverse=True)[:top_k]                   # largest contributions first

scores = model.predict_proba(sample[FEATS])[:, 1]              # fraud score of each sampled row

def make_alert(i):
    fs = factors(i)
    if not fs:
        return (f"Risk score {scores[i]:.2f}. No single factor lies outside the usual range of "
                f"legitimate transfers; the score reflects a combination of factors.")
    parts = [f"{fmt(f, sample.at[i, f], side, share)} (+{c:.2f})" for c, f, side, share in fs]
    return f"Risk score {scores[i]:.2f}. Factors: " + "; ".join(parts) + "."

# 4. Alerts at a 1% false-positive threshold set on validation legitimate rows
v_s = model.predict_proba(va_[FEATS])[:, 1]                    # validation scores
thr = np.quantile(v_s[va_["isFraud"].to_numpy() == 0], 0.99)   # score cut-off for alerts
flagged = np.where(scores > thr)[0]                            # sampled rows that raise an alert
print("Alerts raised in the sample:", len(flagged), "of", len(sample), "\n")

def show(i):
    # The true label and account tier are printed for review only and never appear in the alert text
    print(f"[review only: true label {int(sample.at[i, 'isFraud'])}, "
          f"mixed account {int(sample.at[i, 'acct_mixed'])}]", make_alert(i))

top5 = list(np.argsort(scores)[::-1][:5])                                     # five highest-scoring rows
false5 = [i for i in flagged if sample.at[i, "isFraud"] == 0][:5]             # first five false alarms
rand5 = list(np.random.default_rng(1).choice(flagged, size=min(5, len(flagged)), replace=False))
for i in top5 + false5 + rand5:
    show(i)

# 5. Do true and false alerts cite the same factors? (the key limit for the report)
fl = [factors(i) for i in flagged]                             # citable factors of every alert
summ = pd.DataFrame({
    "label": sample.loc[flagged, "isFraud"].to_numpy(),        # 1 = fraud, 0 = legitimate
    "mixed": sample.loc[flagged, "acct_mixed"].to_numpy(),     # 1 = ordinary account
    "has_factor": [len(x) > 0 for x in fl],                    # at least one citable factor
    "first": [x[0][1] if x else "none" for x in fl],           # largest citable factor
})
print("\nAlerts by label and account tier; share with at least one citable factor:")
print(summ.groupby(["label", "mixed"]).agg(n=("has_factor", "size"), share_with_factor=("has_factor", "mean")).round(3))
print("\nLargest cited factor by label:")
print(pd.crosstab(summ["first"], summ["label"]))

# Save the example alert texts for the report appendix (true label included for review only)
ex = pd.DataFrame({
    "row_id": sample.loc[top5 + false5 + rand5, "row_id"].to_numpy(),
    "true_label": sample.loc[top5 + false5 + rand5, "isFraud"].to_numpy(),
    "alert_text": [make_alert(i) for i in top5 + false5 + rand5],
})
ex.to_csv(f"{OUT}/example_alerts.csv", index=False)

/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Max SHAP reconstruction error: 2.1316282072803006e-14
Mean |SHAP| per feature:
 amt_z_own           0.4174
recip_cnt_24        0.2453
is_new_recipient    0.1270
steps_since_last    0.1028
hour                0.0379
recip_cnt_6         0.0239
dtype: float64 

amt_z_own
                      n  fraud_rate  mean_shap
amt_z_own                                     
(-3.899, -0.268]   2500       0.682     -0.208
(-0.268, -0.236]   2500       0.807      0.457
(-0.236, -0.211]   2500       0.622     -0.261
(-0.211, -0.188]   2500       0.526     -0.501
(-0.188, -0.152]   2500       0.573     -0.349
(-0.152, -0.0746]  2500       0.628     -0.062
(-0.0746, 0.156]   2500       0.706      0.234
(0.156, 28.339]    2500       0.749      0.479

hour
                  n  fraud_rate  mean_shap
hour                                      
(-0.001, 3.0]  3225       0.662      0.027
(3.0, 6.0]     2379       0.675      0.044
(6.0, 9.0]     2335       0.662      0.036
(9.0, 12.0]    2452       0.689      0.0

In [4]:
# Replacement wording helpers; this cell reuses sample, sv, scores, FEATS, factors,
# top5, false5, rand5, flagged and OUT from the explanation cell
import numpy as np
import pandas as pd

def plural(n, word):
    # "1 transfer" versus "3 transfers"
    return f"{n} {word}" if n == 1 else f"{n} {word}s"

def pos_text(side, share):
    # Position of a value among legitimate reference transfers, avoiding the misleading "100%" rounding
    if share < 0.001:                                    # fewer than 0.1% of reference transfers are this extreme
        return "more extreme than any legitimate transfer in the reference data"
    pct = 100 * (1 - share)                              # share of legitimate transfers the value exceeds
    pct_txt = f"{pct:.1f}%" if share < 0.01 else f"{pct:.0f}%"   # one decimal in the extreme tail
    return f"{'higher' if side == 'high' else 'lower'} than {pct_txt} of legitimate transfers"

def fmt(f, v, side, share):
    # States the value and its position among legitimate transfers, with no claim about risk direction
    pos = pos_text(side, share)
    if f == "amt_z_own":
        return f"amount deviation from this account's usual transfer size (z-score {v:+.1f}, {pos})"
    if f == "steps_since_last":
        return f"{plural(int(v), 'step')} since this account's previous transfer ({pos})"
    if f == "recip_cnt_6":
        return f"recipient received {plural(int(v), 'transfer')} in the previous 6 steps ({pos})"
    return f"recipient received {plural(int(v), 'transfer')} in the previous 24 steps ({pos})"   # recip_cnt_24

def make_alert(i):
    # Factors raising the score (outer-tail values only), plus the repeat-recipient factor that lowers it
    fs = factors(i)                                      # citable factors defined in the explanation cell
    if fs:
        parts = [f"{fmt(f, sample.at[i, f], side, share)} (+{c:.2f})" for c, f, side, share in fs]
        text = f"Risk score {scores[i]:.2f}. Factors raising the score: " + "; ".join(parts) + "."
    else:
        text = (f"Risk score {scores[i]:.2f}. No single factor lies outside the usual range of "
                f"legitimate transfers; the score reflects a combination of factors.")
    c_rep = sv[i, FEATS.index("is_new_recipient")]       # contribution of the recipient-novelty feature
    if sample.at[i, "is_new_recipient"] == 0 and c_rep < -0.3:
        text += f" Factor lowering the score: this account had sent to this recipient before ({c_rep:+.2f})."
    return text

# Reprint the same 15 example alerts with the corrected wording (label shown for review only)
for i in top5 + false5 + rand5:
    print(f"[review only: true label {int(sample.at[i, 'isFraud'])}, "
          f"mixed account {int(sample.at[i, 'acct_mixed'])}]", make_alert(i))

# Re-save the appendix examples
sel = top5 + false5 + rand5
pd.DataFrame({
    "row_id": sample.loc[sel, "row_id"].to_numpy(),
    "true_label": sample.loc[sel, "isFraud"].to_numpy(),
    "alert_text": [make_alert(i) for i in sel],
}).to_csv(f"{OUT}/example_alerts.csv", index=False)

# How often does the repeat-recipient factor appear in true and false alerts?
rep = ((sample.loc[flagged, "is_new_recipient"] == 0).to_numpy()
       & (sv[flagged, FEATS.index("is_new_recipient")] < -0.3))
print(pd.crosstab(sample.loc[flagged, "isFraud"].to_numpy(), rep,
                  rownames=["true label"], colnames=["repeat-recipient factor shown"]))

# Edge-effect check: rows in the first 24 steps have a shorter window, which lowers recip_cnt_24
early = sample["step"] < 24                              # rows whose 24-step window is truncated
print("Share of sampled rows in the first 24 steps:", round(float(early.mean()), 3))
low = sample["recip_cnt_24"] <= 7                        # the lowest count bin from the direction table
print(pd.crosstab(low, early, values=sample["isFraud"], aggfunc="mean",
                  rownames=["recip_cnt_24 <= 7"], colnames=["first 24 steps"]).round(3))

[review only: true label 1, mixed account 0] Risk score 0.98. Factors raising the score: amount deviation from this account's usual transfer size (z-score +3.4, higher than 99% of legitimate transfers) (+1.42); 16 steps since this account's previous transfer (more extreme than any legitimate transfer in the reference data) (+1.16).
[review only: true label 1, mixed account 0] Risk score 0.97. Factors raising the score: amount deviation from this account's usual transfer size (z-score +2.9, higher than 98% of legitimate transfers) (+1.57); recipient received 3 transfers in the previous 24 steps (lower than 97% of legitimate transfers) (+0.75).
[review only: true label 1, mixed account 0] Risk score 0.97. Factors raising the score: amount deviation from this account's usual transfer size (z-score +2.6, higher than 98% of legitimate transfers) (+1.38); recipient received 2 transfers in the previous 24 steps (lower than 98% of legitimate transfers) (+0.90).
[review only: true label 1, mixe

In [5]:
# Imports used by this cell
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

OUT = "/kaggle/working"
PREV = 0.01              # realistic fraud prevalence for precision and alert volume
FPRS = [0.01, 0.05]      # target false-positive rates for threshold setting
MIN_STEP = 24            # first step whose 24-step recipient window is complete

# Load the prepared data and the tier definitions saved earlier
data = pd.read_parquet(f"{OUT}/model_data.parquet")
tiers = json.load(open(f"{OUT}/feature_sets.json"))
STRICT = tiers["strict_behavioural"]

# Drift table: row count and fraud share per 24-step block (for the report)
blocks = pd.cut(data["step"], [-1, 23, 47, 71, 95, 119, 143, 167, 192])
print(data.groupby(blocks, observed=True)["isFraud"].agg(rows="size", fraud_share="mean").round(3), "\n")

# Variants: (feature list, minimum step kept)
variants = {
    "A: strict, all steps (reference)": (STRICT, 0),
    "B: strict, steps >= 24": (STRICT, MIN_STEP),
    "C: no recip_cnt_6, steps >= 24": ([f for f in STRICT if f != "recip_cnt_6"], MIN_STEP),
    "D: no recipient counts, steps >= 24": ([f for f in STRICT if not f.startswith("recip_cnt")], MIN_STEP),
}

def fit_lgb(tr_, va_, feats):
    # Gradient-boosted classifier with early stopping on the validation split
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)
    m.fit(tr_[feats], tr_["isFraud"], eval_set=[(va_[feats], va_["isFraud"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(30, verbose=False)])
    return m

def evaluate(v_s, t_s, va, te, fpr_target):
    # Threshold set on validation legitimate rows, then applied to the test scores
    thr = np.quantile(v_s[va["isFraud"].to_numpy() == 0], 1 - fpr_target)
    flag = t_s > thr                                          # alerts raised on the test split
    y = te["isFraud"].to_numpy()
    mixed = te["acct_mixed"].to_numpy() == 1                  # rows from accounts with both classes
    rec = flag[y == 1].mean()                                 # recall over all fraud
    fpr = flag[y == 0].mean()                                 # realised false-positive rate
    alerts = PREV * rec + (1 - PREV) * fpr                    # alerts per transaction at prevalence PREV
    return {"fpr_realised": fpr, "recall_all": rec,
            "recall_fraud_acct_fraud": flag[(y == 1) & ~mixed].mean(),   # fraud in fraud-dedicated accounts
            "recall_mixed_acct_fraud": flag[(y == 1) & mixed].mean(),    # fraud in ordinary accounts
            "alerts_per_1000": 1000 * alerts, "precision_at_1pct": PREV * rec / alerts}

rows = []
for split_col in ["split_time", "split_acct"]:
    for name, (feats, min_step) in variants.items():
        d = data[data["step"] >= min_step]                    # drop early rows where required
        tr_ = d[d[split_col] == "train"]
        va_ = d[d[split_col] == "val"]
        te_ = d[d[split_col] == "test"]
        m = fit_lgb(tr_, va_, feats)
        v_s = m.predict_proba(va_[feats])[:, 1]               # validation scores, used to set thresholds
        t_s = m.predict_proba(te_[feats])[:, 1]               # test scores
        auc = roc_auc_score(te_["isFraud"], t_s)              # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)
            r.update({"split": split_col, "variant": name, "fpr_target": fpr,
                      "n_test": len(te_), "roc_auc": auc})
            rows.append(r)

cols = ["split", "variant", "fpr_target", "n_test", "roc_auc", "fpr_realised", "recall_all",
        "recall_fraud_acct_fraud", "recall_mixed_acct_fraud", "alerts_per_1000", "precision_at_1pct"]
print(pd.DataFrame(rows)[cols].round(3).to_string(index=False))

              rows  fraud_share
step                           
(-1, 23]     58788        0.908
(23, 47]     89368        0.781
(47, 71]    108111        0.699
(71, 95]    107279        0.695
(95, 119]   107018        0.681
(119, 143]  110701        0.656
(143, 167]  125701        0.602
(167, 192]  140744        0.559 

     split                             variant  fpr_target  n_test  roc_auc  fpr_realised  recall_all  recall_fraud_acct_fraud  recall_mixed_acct_fraud  alerts_per_1000  precision_at_1pct
split_time    A: strict, all steps (reference)        0.01  216273    0.646         0.008       0.042                    0.057                    0.008            7.929              0.052
split_time    A: strict, all steps (reference)        0.05  216273    0.646         0.027       0.077                    0.100                    0.027           27.429              0.028
split_time              B: strict, steps >= 24        0.01  216273    0.648         0.007       0.042             

In [6]:
# Imports used by this cell
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

OUT = "/kaggle/working"
MIN_STEP = 24            # first step whose 24-step recipient window is complete
PREV = 0.01              # realistic fraud prevalence for precision and alert volume
FPRS = [0.01, 0.05]      # target false-positive rates for threshold setting
N_BOOT = 200             # account-level bootstrap resamples (lower to 100 if the cell is slow)

# Load the prepared data and keep one common scope for every method
data = pd.read_parquet(f"{OUT}/model_data.parquet")
data = data[data["step"] >= MIN_STEP].reset_index(drop=True)
print("Rows in final scope:", len(data))

# Final tier definitions; profile and balance tiers are carried over from the saved file
old = json.load(open(f"{OUT}/feature_sets.json"))
STRICT = ["amt_z_own", "hour", "steps_since_last", "is_new_recipient", "recip_cnt_24"]
tiers = {
    "strict_behavioural": STRICT,
    "behavioural_plus_amount": STRICT + ["amount_log"],
    "profile": old["profile"],
    "balance": old["balance"],
}
with open(f"{OUT}/feature_sets.json", "w") as fh:
    json.dump(tiers, fh, indent=2)                            # the explanation cell reads this file

def fit_lgb(tr_, va_, feats):
    # Gradient-boosted classifier with early stopping on the validation split
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=0, verbose=-1)
    m.fit(tr_[feats], tr_["isFraud"], eval_set=[(va_[feats], va_["isFraud"])],
          eval_metric="auc", callbacks=[lgb.early_stopping(30, verbose=False)])
    return m

def metrics(flag, y, mixed, unusual):
    # Metrics for one set of alert flags on one set of test rows
    rec = flag[y == 1].mean()                                 # recall over all fraud
    fpr = flag[y == 0].mean()                                 # false-positive rate over legitimate rows
    alerts = PREV * rec + (1 - PREV) * fpr                    # alerts per transaction at prevalence PREV
    return {
        "recall_all": rec,
        "recall_fraud_acct_fraud": flag[(y == 1) & ~mixed].mean(),   # fraud in fraud-dedicated accounts
        "recall_mixed_acct_fraud": flag[(y == 1) & mixed].mean(),    # fraud in ordinary accounts
        "fpr": fpr,
        "flag_rate_unusual_legit": flag[unusual].mean() if unusual.any() else np.nan,
        "precision_at_1pct": PREV * rec / alerts if alerts > 0 else np.nan,
        "alerts_per_1000": 1000 * alerts,
    }

def evaluate(v_s, t_s, va, te, fpr_target):
    # Threshold set on validation legitimate rows, then applied to the test scores
    thr = np.quantile(v_s[va["isFraud"].to_numpy() == 0], 1 - fpr_target)
    flag = t_s > thr                                          # alerts raised on the test split
    y = te["isFraud"].to_numpy()
    mixed = te["acct_mixed"].to_numpy() == 1
    unusual = te["stratum_legit_unusual"].to_numpy() == 1
    out = metrics(flag, y, mixed, unusual)

    # Account-level bootstrap: resample test accounts with replacement, recompute recall and FPR
    acc = te["acct_id"].to_numpy()
    idx_by_acct = pd.Series(np.arange(len(te))).groupby(acc).apply(np.asarray)   # row positions per account
    rng = np.random.default_rng(0)
    boots = []
    for _ in range(N_BOOT):
        pick = rng.choice(len(idx_by_acct), size=len(idx_by_acct), replace=True)  # resample accounts
        rows_ = np.concatenate(idx_by_acct.iloc[pick].to_numpy())                 # all rows of the picked accounts
        b = metrics(flag[rows_], y[rows_], mixed[rows_], unusual[rows_])
        boots.append((b["recall_all"], b["fpr"]))
    boots = np.array(boots)
    out["recall_all_lo"], out["recall_all_hi"] = np.nanpercentile(boots[:, 0], [2.5, 97.5])
    out["fpr_lo"], out["fpr_hi"] = np.nanpercentile(boots[:, 1], [2.5, 97.5])
    return out

rows = []
for split_col in ["split_time", "split_acct"]:
    # Training, validation and test rows for this split definition
    tr_ = data[data[split_col] == "train"]
    va_ = data[data[split_col] == "val"]
    te_ = data[data[split_col] == "test"]
    print(split_col, "test rows:", len(te_))

    scores = {}   # method name -> (validation scores, test scores); higher means more suspicious

    # Baselines: fixed amount rule, own-norm deviation rule, Isolation Forest on the strict tier
    scores["rule: amount"] = (va_["amount_log"].to_numpy(), te_["amount_log"].to_numpy())
    scores["rule: own-norm"] = (va_["amt_z_own"].abs().to_numpy(), te_["amt_z_own"].abs().to_numpy())
    iso = IsolationForest(n_estimators=200, max_samples=50_000, random_state=0, n_jobs=-1)
    iso.fit(tr_.loc[tr_["isFraud"] == 0, STRICT])             # fitted on legitimate rows only
    scores["isolation forest"] = (-iso.score_samples(va_[STRICT]), -iso.score_samples(te_[STRICT]))

    # Supervised models, one per final tier; the strict-tier model is saved for the explanation layer
    for name, feats in tiers.items():
        m = fit_lgb(tr_, va_, feats)
        scores[f"lgbm: {name}"] = (m.predict_proba(va_[feats])[:, 1], m.predict_proba(te_[feats])[:, 1])
        if name == "strict_behavioural":
            joblib.dump(m, f"{OUT}/model_strict_{split_col}.joblib")

    # Evaluate every method at every target false-positive rate
    for method, (v_s, t_s) in scores.items():
        auc = roc_auc_score(te_["isFraud"], t_s)              # threshold-free ranking quality
        for fpr in FPRS:
            r = evaluate(v_s, t_s, va_, te_, fpr)
            r.update({"split": split_col, "method": method, "fpr_target": fpr, "roc_auc": auc})
            rows.append(r)

# Collect, print and save the results table
cols = ["split", "method", "fpr_target", "roc_auc", "fpr", "fpr_lo", "fpr_hi",
        "recall_all", "recall_all_lo", "recall_all_hi",
        "recall_fraud_acct_fraud", "recall_mixed_acct_fraud",
        "flag_rate_unusual_legit", "alerts_per_1000", "precision_at_1pct"]
res = pd.DataFrame(rows)[cols]
res.to_csv(f"{OUT}/final_results.csv", index=False)
print(res.round(3).to_string(index=False))

Rows in final scope: 788922
split_time test rows: 216273
split_acct test rows: 155176
     split                        method  fpr_target  roc_auc   fpr  fpr_lo  fpr_hi  recall_all  recall_all_lo  recall_all_hi  recall_fraud_acct_fraud  recall_mixed_acct_fraud  flag_rate_unusual_legit  alerts_per_1000  precision_at_1pct
split_time                  rule: amount        0.01    0.485 0.018   0.017   0.020       0.000          0.000          0.000                    0.000                    0.000                    0.716           18.023              0.000
split_time                  rule: amount        0.05    0.485 0.057   0.046   0.070       0.030          0.021          0.038                    0.030                    0.029                    0.990           56.754              0.005
split_time                rule: own-norm        0.01    0.561 0.016   0.015   0.017       0.000          0.000          0.000                    0.000                    0.000                    0.683   

In [9]:
# Keep the same scope as the final evaluation (complete recipient windows)
data = data[data["step"] >= 24].reset_index(drop=True)